# Perceptrón simple de Rosenblatt — Implementación y simulación desde cero

**Repositorio del proyecto:** <https://github.com/022100127h-ship-it/perceptron.git>
(`git clone https://github.com/022100127h-ship-it/perceptron.git`)

> Este notebook (`perceptron.ipynb`) **forma parte del proyecto `perceptron`** y se integra con su
> código base: los archivos `perceptron.py` y `utils.py` del repositorio. En la
> sección de integración se carga ese código con
> `importlib`, se comparan sus resultados con los de la clase implementada aquí y se explica
> qué aporta cada uno. Si clonas el repositorio y abres el notebook desde la raíz, la
> integración se activa automáticamente.

---

## Objetivo

Implementar **desde cero** la neurona de Frank Rosenblatt (1957-1962) usando **únicamente NumPy**
para el cálculo y **Matplotlib** para la visualización, y estudiar su comportamiento en el
problema clásico de la separabilidad lineal: las compuertas lógicas **AND**, **OR** y **XOR**.

**Restricción del enunciado:** está prohibido usar `scikit-learn` u cualquier otro framework de
IA. No se importa ninguno: todo el álgebra (producto escalar, producto matriz-vector,
actualización de pesos, métricas) está escrito a mano, y la sección 13 lo verifica automáticamente
releyendo el código de este mismo notebook.

## Objetivos de aprendizaje

1. Comprender la arquitectura matemática interna de una neurona artificial (Perceptrón de Rosenblatt).
2. Implementar el algoritmo de entrenamiento usando solo álgebra vectorial básica con NumPy.
3. Simular y evaluar el comportamiento en problemas de separación lineal (AND, OR, XOR).
4. Demostrar geométricamente la limitación fundamental del perceptrón de una sola capa.

## Índice

| # | Sección | Contenido |
|---|---------|-----------|
| 0 | Fundamento teórico | ecuaciones de la neurona y regla de aprendizaje |
| 1 | La clase `Perceptron` | implementación con NumPy |
| 2 | Prueba de humo | validación de la API y casos borde |
| 3 | Conjuntos de datos | AND, OR, XOR y diagnóstico de separabilidad |
| 4 | Experimento 1 — AND | entrenamiento y análisis |
| 5 | Experimento 2 — OR | entrenamiento y análisis |
| 6 | Experimento 3 — XOR | entrenamiento y análisis |
| 7 | Comparativa global | figura resumen 2x3 de los tres experimentos |
| 8 | XOR: por qué es imposible | cascos convexos, cota del 75 % y punto fijo |
| 9 | Ampliación: XOR con capa oculta | mapa no lineal de características |
| 10 | Verificación científica de $\eta$ | proporcionalidad $w = \eta\hat{w}$ y aritmética exacta |
| 11 | Integración con el repositorio | carga de `perceptron.py` y `utils.py` |
| 12 | Cuestionario (Parte 4) | las tres preguntas con respuesta fundamentada |
| 13 | Cumplimiento del enunciado | verificación automática de la rúbrica |
| 14 | Conclusiones y referencias | síntesis y cómo subir el notebook al repo |

## Requisitos y ejecución

```bash
pip install numpy matplotlib jupyter
jupyter lab perceptron.ipynb     # o: jupyter notebook perceptron.ipynb
```

El notebook se ejecuta **de arriba abajo sin errores**: todas las figuras se guardan además en la
carpeta `figures/` y todas las cifras citadas en el cuestionario se imprimen en celdas de código
antes de ser comentadas, de modo que el texto y la salida nunca pueden desincronizarse.

In [ ]:
# =============================================================================
# Celda 2 - Configuracion del entorno
# Solo NumPy (calculo) y Matplotlib (graficas). Ningun framework de IA.
# =============================================================================
%matplotlib inline

import os
import sys
import json
import pathlib
import importlib.util as importlib_util
from typing import Dict, List, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- Presentacion consistente de los objetos numpy en las tablas ---
np.set_printoptions(precision=4, suppress=True, linewidth=120)

# --- Configuracion global de los experimentos ---
EPOCHS = 100                      # numero maximo de epocas (parada temprana si converge)
LEARNING_RATES = [0.01, 0.1, 0.5] # tres tasas de aprendizaje a comparar
COLORS = {0.01: "tab:blue", 0.1: "tab:green", 0.5: "tab:red"}

# --- Carpeta donde se guardan las figuras (no se sobreescriben los PNG del repo) ---
FIG_DIR = pathlib.Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# --- Estilo global de las figuras ---
matplotlib.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Evidencia de que no estamos usando scikit-learn ni ningun otro framework de ML ---
FORBIDDEN = ("sklearn", "scikit_learn", "torch", "tensorflow", "keras", "jax", "pandas")

def check_no_ml_libraries() -> bool:
    """Verifica que ninguna libreria de ML/IA este instalada *usada* por este notebook.

    Se inspeccionan los modulos ya importados: si un framework de ML apareciera en el
    grafo de imports de este kernel, la funcion lo detectaria y devolveria False.
    """
    loaded = set(sys.modules)
    offenders = [name for name in loaded if name.split(".")[0] in FORBIDDEN]
    assert not offenders, f"Se importaron frameworks de IA no permitidos: {offenders}"
    return True

assert check_no_ml_libraries(), "El notebook debe usar solo NumPy + Matplotlib"

print("=" * 78)
print("  PERCEPTRON DE ROSENBLATT - CONFIGURACION DEL ENTORNO")
print("=" * 78)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  NumPy       : {np.__version__}")
print(f"  Matplotlib  : {matplotlib.__version__}")
print(f"  Epocas max  : {EPOCHS}")
print(f"  Tasas (eta) : {LEARNING_RATES}")
print(f"  Figuras en  : {FIG_DIR.resolve()}")
print("  Librerias de IA importadas: ninguna (verificado)")
print("=" * 78)

## 0. Fundamento teórico

### 0.1 La neurona: sumas, pesos y un escalón

El perceptrón de Rosenblatt es una unidad de cómputo que recibe un vector de entrada
$\mathbf{x} = [x_1, x_2, \dots, x_n]^\top$, un vector de pesos
$\mathbf{w} = [w_1, w_2, \dots, w_n]^\top$ y un **sesgo** $b$ (también llamado *bias* u
*umbral*), y devuelve una salida binaria. La operación tiene dos etapas:

**Etapa 1 — combinación lineal (potencial de activación, o *net input*):**

$$
z = \sum_{i=1}^{n} w_i\,x_i + b = \mathbf{w}^\top \mathbf{x} + b
$$

Equivalente, $\mathbf{w}^\top\mathbf{x} = \mathbf{w}\cdot\mathbf{x}$ es el **producto escalar**
y, para una matriz de datos $X \in \mathbb{R}^{m\times n}$ con una fila por muestra, todos los
$z$ de golpe se obtienen con un único **producto matriz-vector**: $\mathbf{z} = X\mathbf{w} + b$ (el
sesgo se suma por *broadcasting*). Esta es la operación matricial central del modelo.

**Etapa 2 — activación no lineal (función escalón de Heaviside):**

$$
\hat{y} = f(z) = \begin{cases} 1 & \text{si } z \ge 0 \\ 0 & \text{si } z < 0 \end{cases}
$$

> **Convenio de este notebook:** el punto $z = 0$ se asigna a la **clase 1** (`z >= 0`).
> La elección importa, porque con pesos iniciales nulos ($z = 0$ en la primera predicción) decide
> qué se considera "ya correcto" al arrancar. Es la convención canónica de Rosenblatt y la que
> usa el enunciado.

La frontera de decisión es el conjunto de puntos donde la salida cambia de clase:

$$
\mathbf{w}^\top \mathbf{x} + b = 0 \quad\Longleftrightarrow\quad
w_1 x_1 + w_2 x_2 + b = 0
$$

Despejando $x_2$ (que es como se dibuja en las gráficas de la Parte 3):

$$
\boxed{\;x_2 = -\frac{w_1}{w_2}\,x_1 - \frac{b}{w_2}\;}
$$

Si $w_2 = 0$ la recta es vertical ($x_1 = -b/w_1$) y la fórmula anterior no es aplicable: el
código de graficado lo detecta y dibuja ese caso. Si $\mathbf{w} = \mathbf{0}$ y $b = 0$ la
frontera es *degenerada* (la neurona no depende de la entrada).

### 0.2 Regla de aprendizaje

Para cada muestra $i$ se compara la salida real con la predicción y se define el **error**:

$$
e = y^{(i)} - \hat{y}^{(i)} \in \{-1,\, 0,\, +1\}
$$

Si $e \neq 0$ la muestra está mal clasificada y se corrigen los parámetros en la dirección que
reduce ese error (con $\eta$ la *tasa de aprendizaje*, $0 < \eta \le 1$):

$$
w_j \leftarrow w_j + \eta \cdot \big(y^{(i)} - \hat{y}^{(i)}\big) \cdot x_j^{(i)},
\qquad j = 1 \dots n
$$

$$
b \leftarrow b + \eta \cdot \big(y^{(i)} - \hat{y}^{(i)}\big)
$$

Obsérvese que el sesgo se actualiza con la **misma** regla: es como si la neurona tuviera una
entrada extra constante $x_0 = 1$ con peso $b$. Por eso, formalmente, el perceptrón opera en el
espacio aumentado $\tilde{\mathbf{x}} = [x_1,\dots,x_n, 1]$ y el teorema de Cover se aplica sin
cambios.

**Pseudocódigo (una época = una pasada completa por el dataset, actualización *online*):**

```text
para epoca = 1 .. epochs:                 # como maximo 'epochs' epocas
    total_errors = 0
    para i = 1 .. n_muestras:             # modo online: se corrige en el acto
        z     = w . x[i] + b              # producto escalar
        y_hat = 1 si z >= 0 sino 0        # activacion escalon
        e     = y[i] - y_hat              # error
        si e != 0:                        # solo se corrige lo mal clasificado
            w = w + eta * e * x[i]
            b = b + eta * e
            total_errors += 1
    registrar total_errors en errors_history
    si total_errors == 0:                 # ¡convergido!
        imprimir "convergió en la época epoca"
        break                             # parada temprana
```

La actualización es *aditiva y por muestra*: el perceptrón **no minimiza una función de pérdida
como la entropía cruzada**; simplemente "empuja" el vector de pesos (normalmente un
multiplicador de $\mathbf{x}^{(i)}$) para clasificar bien esa muestra.

### 0.3 Teorema de convergencia y su condición

> **Teorema (Cover / Rosenblatt).** Si los datos son **linealmente separables** y la tasa de
> aprendizaje es acotada ($0 < \eta \le 1$), el perceptrón converge en un número finito de pasos,
> con cota
> $$T \;\le\; \frac{\beta^2}{\eta^2\gamma^2}\Big(\max_i\|\tilde{\mathbf{x}}^{(i)}\|^2 + R^2\Big)$$
> donde $\gamma$ es el margen de separación, $\beta = \max_i\|y^{(i)}\|$ y $R = \max_i\|\tilde{\mathbf{x}}^{(i)}\|$.

Dos consecuencias que **se observarán experimentalmente** en este notebook:

* Si los datos **no** son separables linealmente, el algoritmo **no converge nunca**: la frontera
  oscila indefinidamente. No es un fallo de implementación, es una limitación del modelo.
* El margen $\gamma$ aparece en el denominador: cuanto mayor es el margen, **más rápido** converge
  el algoritmo. Como el perceptrón se detiene en cuanto hay 0 errores (no maximiza el margen),
  suele terminar en una solución *degenerada* (una muestra justo sobre la frontera), no en la de
  máximo margen.

### 0.4 Por qué una sola capa es insuficiente

Un perceptrón con $n$ entradas solo puede separar clases con una **recta** en $\mathbb{R}^n$. El
problema XOR necesita una frontera de decisión que no es una recta, y de ahí la famosa demostración de
Minsky & Papert (1969) que motivó las redes multicapa. La Parte 4 analiza ese límite con gráficos
y la sección 9 muestra cómo una única
capa oculta lo resuelve.

## 1. La clase `Perceptron`

Implementación completa en NumPy. Decisiones de diseño y su justificación:

| Decisión | Motivo |
|----------|--------|
| Pesos en ceros (`init="zeros"`) con opción `init="random"` | Con ceros el modelo parte de la frontera degenerada $z=0$; es la inicialización canónica de Rosenblatt y hace el experimento reproducible. La aleatoria pequeña sirve para mostrar que el resultado final no depende (salvo empates numéricos) del punto de partida. |
| Activación `1 si z >= 0` | Convención del enunciado; el caso $z=0$ se asigna a la clase 1. |
| `errors_history` = correcciones por época | Es el "número total de errores" que pide el enunciado: cuantas muestras fueron detectadas mal clasificadas en cada pasada. Con actualización *online* una muestra corregida puede volver a fallar, así que este contador **no es necesariamente** el número de errores del modelo al final de la época (por eso se registra además `accuracy_history`). |
| `accuracy_history` | Exactitud (accuracy) medida al inicio de cada época, con un solo producto matriz-vector `X @ w + b`: la lectura de la convergencia no depende del desajuste anterior. |
| Parada temprana con `break` | Si `total_errors == 0` no hay nada que corregir; seguir iterando es tiempo perdido. |
| `fit` *online* + `fit_bulk` vectorizado | `fit` es el algoritmo de Rosenblatt muestra a muestra (el del enunciado y el del repo). `fit_bulk` muestra la variante 100 % matricial (todas las correcciones de la época en una sola operación `X[bad].T @ e[bad]`), útil para cumplir el requisito de *matriciado adecuado*. |
| `predict` acepta vector o matriz | `predict(x)` con `x` 1-D devuelve un `int` (una neurona, una entrada); con `x` 2-D devuelve un vector de predicciones, que es lo que necesitan `score` y las gráficas. |
| Validación de hiperparámetros | `0 < eta <= 1` es el intervalo del teorema de convergencia: se hace explícito en el constructor. |

In [ ]:
# =============================================================================
# PARTE 1 - LA CLASE PERCEPTRON (desde cero, solo NumPy)
# =============================================================================
import numpy as np


class Perceptron:
    """Perceptron de Rosenblatt: clasificador lineal binario implementado desde cero.

    Modelo
    ------
        z      = w . x + b                (combinacion lineal, producto escalar)
        y_hat  = 1 si z >= 0, 0 si z < 0  (activacion escalon de Heaviside)

    Aprendizaje (regla de Rosenblatt, modo *online*, muestra a muestra)
    --------------------------------------------------------------------
        e = y - y_hat
        si e != 0:
            w <- w + eta * e * x
            b <- b + eta * e

    Solo se usa algebra lineal basica de NumPy: producto escalar, producto
    matriz-vector y suma ponderada de filas. No hay ninguna libreria de ML.

    Notas teoricas
    --------------
    * Si los datos son linealmente separables, el algoritmo converge en un numero
      finito de pasos (teorema de Cover/Rosenblatt).
    * Si NO son separables, nunca converge: la frontera oscila para siempre.
    * Al detenerse en el primer 0 errores, la solucion suelen ser *degenerada*
      (una muestra sobre la frontera), no la de maximo margen.
    """

    def __init__(self, input_size: int, learning_rate: float = 0.1, epochs: int = 100,
                 init: str = "zeros", random_state: int | None = None, verbose: bool = True):
        """Inicializa la neurona con sus hiperparametros y sus parametros de decision.

        Parametros
        -----------
        input_size : int
            Numero de caracteristicas de entrada n (dimension de w).
        learning_rate : float, por defecto 0.1
            Tasa de aprendizaje eta, con 0 < eta <= 1. Controla el tamano del
            paso de cada correccion: los pesos convergen a un multiplo de eta
            (w = eta * w*), de modo que la *forma* de la frontera no depende de
            eta y solo cambia la escala del margen.
        epochs : int, por defecto 100
            Numero maximo de epocas (pasadas completas sobre los datos). El
            entrenamiento se detiene antes si converge (parada temprana).
        init : {"zeros", "random"}, por defecto "zeros"
            "zeros"  -> w = [0, 0, ..., 0] (inicializacion canonica).
            "random" -> w ~ U(-0.1, 0.1) usando `random_state` (reproducible).
        random_state : int | None
            Semilla del generador de numeros aleatorios (solo si init="random").
        verbose : bool, por defecto True
            Si True, `fit` imprime el mensaje de convergencia o de fracaso.

        Atributos
        ---------
        w : np.ndarray de forma (n,)
            Vector de pesos.
        b : float
            Sesgo (bias) u umbral.
        errors_history : list[int]
            Correcciones (muestras mal clasificadas detectadas) por epoca.
        accuracy_history : list[float]
            Exactitud al inicio de cada epoca.
        n_iter_ : int
            Epocas realmente ejecutadas.
        converged_ : bool
            True si se alcanzó 0 errores (convergio).
        """
        if int(input_size) <= 0:
            raise ValueError("input_size debe ser un entero positivo.")
        if not (0.0 < float(learning_rate) <= 1.0):
            raise ValueError("learning_rate (eta) debe cumplir 0 < eta <= 1.")

        self.input_size = int(input_size)
        self.learning_rate = float(learning_rate)
        self.epochs = int(epochs)
        self.init = init
        self.random_state = random_state
        self.verbose = verbose

        # Pesos iniciales: ceros (canonico) o pequenos valores aleatorios.
        if init == "zeros":
            self.w = np.zeros(self.input_size, dtype=float)
        elif init == "random":
            rng = np.random.default_rng(random_state)
            self.w = rng.uniform(-0.1, 0.1, self.input_size)
        else:
            raise ValueError("init debe ser 'zeros' o 'random'.")

        # El sesgo arranca en 0.0 y errors_history recoge los errores por epoca.
        self.b = 0.0
        self.errors_history: List[int] = []
        self.accuracy_history: List[float] = []

        self.n_iter_ = 0
        self.converged_ = False
        self._fitted = False

    # -------------------------------------------------------------------------
    # Etapa 2 del modelo: activacion
    # -------------------------------------------------------------------------
    def activation_function(self, z):
        """Funcion escalon de Heaviside: 1 si z >= 0, 0 en caso contrario.

        Es la unica no linealidad del modelo. Se implementa con `np.where`, que
        opera vectorialmente: acepta un escalar o un array y devuelve un entero o
        un array de enteros con la misma forma que la entrada.

        El caso z == 0 se asigna a la clase 1 (convencion de Rosenblatt usada en
        todo el notebook).
        """
        return np.where(np.asarray(z, dtype=float) >= 0, 1, 0)

    # -------------------------------------------------------------------------
    # Etapa 1 del modelo: combinacion lineal
    # -------------------------------------------------------------------------
    def decision_function(self, X):
        """Calcula z = w . x + b, el potencial de activacion (pre-activacion).

        Acepta:
          * `X` de forma (n,)   -> un escalar z = w . x + b (una sola muestra).
          * `X` de forma (m, n) -> un array de forma (m,), calculado con un unico
            producto matriz-vector `X @ w`, mas la suma del sesgo por broadcasting.
        """
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            if X.shape[0] != self.input_size:
                raise ValueError(
                    f"Se esperaban {self.input_size} caracteristicas, se recibieron {X.shape[0]}.")
            return float(np.dot(X, self.w) + self.b)          # producto escalar
        if X.ndim == 2:
            if X.shape[1] != self.input_size:
                raise ValueError(
                    f"Se esperaban {self.input_size} caracteristicas, se recibieron {X.shape[1]}.")
            return X @ self.w + self.b                        # producto matriz-vector
        raise ValueError("X debe ser un vector (n,) o una matriz (m, n).")

    # -------------------------------------------------------------------------
    # Prediccion
    # -------------------------------------------------------------------------
    def predict(self, x):
        """Predice la clase a partir de la activacion escalon de la pre-activacion.

        Parametros
        ----------
        x : array de forma (n,) o (m, n)
            Una muestra (devuelve un `int`) o un lote de muestras (devuelve un
            array de `int` de forma (m,)).

        Devuelve
        --------
        int | np.ndarray
            1 si z >= 0, 0 si z < 0.
        """
        z = self.decision_function(x)
        if np.ndim(z) == 0:                    # una sola muestra -> escalar
            return int(self.activation_function(z))
        return self.activation_function(z)     # lote -> array vectorizado

    # -------------------------------------------------------------------------
    # Entrenamiento
    # -------------------------------------------------------------------------
    def fit(self, X, y) -> "Perceptron":
        """Entrena la neurona con el algoritmo de aprendizaje de Rosenblatt.

        Se recorre el dataset muestra a muestra (modo *online*): la correccion de
        una muestra se aplica inmediatamente, antes de evaluar la siguiente. Este
        es el algoritmo del enunciado y el del repositorio.

        En cada epoca:
          1. Se registra la exactitud actual con un producto matriz-vector.
          2. Para cada muestra i se calcula z = w . x[i] + b y y_hat = f(z).
          3. Se calcula el error e = y[i] - y_hat. Si e != 0:
                 w <- w + eta * e * x[i]
                 b <- b + eta * e
          4. Se acumula el numero de correcciones en `errors_history`.
          5. Si el total fue 0, el modelo ya clasifica bien todo el dataset: se
             imprime el mensaje de convergencia y se hace `break` (parada temprana).

        Devuelve `self` para poder encadenar (`Perceptron(...).fit(X, y)`).
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).astype(int).ravel()
        if X.ndim != 2 or X.shape[0] != y.shape[0]:
            raise ValueError("X debe tener forma (m, n) e y debe tener m elementos.")

        n_samples = X.shape[0]
        self.errors_history, self.accuracy_history = [], []
        self.converged_, self.n_iter_ = False, 0

        for epoch in range(self.epochs):
            # (1) Exactitud al inicio de la epoca: una sola operacion matricial.
            self.accuracy_history.append(
                float(np.mean(self.activation_function(self.decision_function(X)) == y)))

            # (2-3) Pasada online: pre-activacion, prediccion, error y correccion.
            total_errors = 0
            for i in range(n_samples):
                z_i = float(np.dot(X[i], self.w) + self.b)     # producto escalar w . x[i]
                y_hat_i = int(self.activation_function(z_i))     # activacion escalon
                error = int(y[i]) - y_hat_i                     # e = y - y_hat
                if error != 0:                                  # solo se corrige si falla
                    self.w += self.learning_rate * error * X[i]   # w <- w + eta * e * x[i]
                    self.b += self.learning_rate * error           # b <- b + eta * e
                    total_errors += 1

            # (4) Registro del numero de errores de la epoca.
            self.errors_history.append(total_errors)
            self.n_iter_ = epoch + 1

            # (5) Parada temprana: 0 errores = frontera valida para todo el dataset.
            if total_errors == 0:
                self.converged_ = True
                if self.verbose:
                    print(f"[OK] Convergencia alcanzada en la epoca {epoch + 1} (0 errores).")
                    print(f"     Pesos finales : w = {np.round(self.w, 4)}   b = {self.b:.4f}")
                break
        else:
            # El `for` finishes sin break: se agotaron las epocas.
            if self.verbose:
                print(f"[X] No se alcanzo la convergencia en {self.epochs} epocas.")
                print(f"    Errores en la ultima epoca: {self.errors_history[-1]}"
                      f" | w = {np.round(self.w, 4)} | b = {self.b:.4f}")

        self._fitted = True
        return self

    def fit_bulk(self, X, y) -> "Perceptron":
        """Variante 100 % vectorizada del entrenamiento (correccion simultanea).

        En lugar de recorrer las muestras con un bucle, se calculan *todas* las
        correcciones de la epoca con dos operaciones de algebra lineal:

            e      = y - f(X @ w + b)        # errores de toda la epoca (vector)
            idx    = flatnonzero(e != 0)     # indices de las muestras mal clasificadas
            w     += eta * (X[idx].T @ e[idx])   # una sola multiplicacion matricial
            b     += eta * e[idx].sum()          # una sola suma

        Es más rápida, pero NO es el algoritmo de Rosenblatt: corrige con los
        errores "congelados" al inicio de la época, no después de cada muestra.
        Se incluye para contrastar ambos órdenes de actualización y para mostrar el
        uso matricial del enunciado. Para datos separables ambos convergen a una
        solución equivalente; pueden diferir en el número de épocas.
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).astype(int).ravel()
        self.errors_history, self.accuracy_history = [], []
        self.converged_, self.n_iter_ = False, 0

        for epoch in range(self.epochs):
            y_hat = self.activation_function(self.decision_function(X))   # X @ w + b
            self.accuracy_history.append(float(np.mean(y_hat == y)))

            errors = y - y_hat
            bad = np.flatnonzero(errors != 0)          # muestras mal clasificadas
            self.errors_history.append(int(bad.size))
            self.n_iter_ = epoch + 1

            if bad.size == 0:                          # 0 errores -> convergido
                self.converged_ = True
                if self.verbose:
                    print(f"[OK] Convergencia alcanzada en la epoca {epoch + 1} (0 errores).")
                break

            # Correccion simultanea de todas las muestras mal clasificadas.
            self.w += self.learning_rate * (X[bad].T @ errors[bad].astype(float))
            self.b += self.learning_rate * float(errors[bad].sum())

        if not self.converged_ and self.verbose:
            print(f"[X] No se alcanzo la convergencia en {self.epochs} epocas.")
        self._fitted = True
        return self

    # -------------------------------------------------------------------------
    # Utilidades
    # -------------------------------------------------------------------------
    def score(self, X, y) -> float:
        """Exactitud (accuracy) = fraccion de muestras correctamente clasificadas.

        Se calcula vectorialmente: una prediccion por producto matriz-vector y una
        comparacion contra las etiquetas reales.
        """
        return float(np.mean(self.predict(X) == np.asarray(y).ravel()))

    def summary(self) -> Dict:
        """Devuelve un diccionario con los resultados clave del entrenamiento."""
        return {
            "w": self.w.copy(),
            "b": float(self.b),
            "learning_rate": self.learning_rate,
            "n_iter": self.n_iter_,
            "converged": self.converged_,
            "errors_history": list(self.errors_history),
            "accuracy_final": (self.accuracy_history[-1] if self.accuracy_history else float("nan")),
        }

    def __repr__(self) -> str:
        return (f"Perceptron(weights={np.round(self.w, 4)}, bias={self.b:.4f}, "
                f"learning_rate={self.learning_rate}, epochs={self.epochs}, "
                f"fitted={self._fitted}, converged={self.converged_})")

## 2. Prueba de humo de la clase

Antes de los experimentos se comprueba, de forma explícita, que:

1. `activation_function` respeta el convenio $z \ge 0 \Rightarrow 1$ **en el borde** (incluidos
   `-0.0` y el entero `0` de NumPy, que no son negativos a efectos de comparación).
2. `predict` devuelve un `int` para una muestra suelta y un array para un lote, y ambos
   coinciden con la fórmula $f(\mathbf{w}^\top\mathbf{x}+b)$ calculada a mano.
3. `fit` entrena, acumula `errors_history`, imprime la convergencia y hace parada temprana
   (`len(errors_history) < epochs`).
4. La variante matricial `fit_bulk` produce la misma exactitud que `fit`.
5. El constructor rechaza hiperparámetros fuera de rango.

In [ ]:
# =============================================================================
# PARTE 1 (continuacion) - Prueba de humo de la clase
# =============================================================================
print("--- 1) activation_function en los casos borde -------------------------")
p = Perceptron(2, learning_rate=0.1, epochs=100, verbose=False)
for z in [5.0, 0.1, 0.0, -0.0, -0.1, -5.0]:
    print(f"    f({z:>5}) = {p.activation_function(z)}")
assert p.activation_function(0.0) == 1 and p.activation_function(-0.0) == 1
assert p.activation_function(np.array([1.0, -1.0, 0.0])).tolist() == [1, 0, 1]
print("    OK: el borde z = 0 pertenece a la clase 1 (tambien para -0.0).")

print("\n--- 2) predict: una muestra (int) vs. lote (array) ------------------")
p.w = np.array([0.5, 0.5]); p.b = -0.5
print(f"    w = {p.w}, b = {p.b}")
for x in [[0, 0], [0, 1], [1, 0], [1, 1]]:
    z_manual = 0.5 * x[0] + 0.5 * x[1] - 0.5
    print(f"    x={x}  z={z_manual:+.2f}  predict(x)={p.predict(x)}  (esperado {int(z_manual >= 0)})")
assert isinstance(p.predict([1, 1]), int)                      # una muestra -> int
assert p.predict([[0, 0], [1, 1]]).tolist() == [0, 1]         # lote -> array
assert p.decision_function([1, 1]) == 0.5                     # z = 0.5 + 0.5 - 0.5
assert p.decision_function([[0, 0], [1, 1]]).tolist() == [-0.5, 0.5]   # producto matriz-vector
print("    OK: salida escalar para una muestra, vectorial para un lote.")

print("\n--- 3) fit sobre AND: convergencia y parada temprana ---------------")
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
p = Perceptron(2, learning_rate=0.1, epochs=100, verbose=True)
p.fit(X_and, y_and)
print(f"    errors_history  = {p.errors_history}")
print(f"    accuracy_history= {[round(a, 3) for a in p.accuracy_history]}")
print(f"    epocas ejecutadas = {p.n_iter_} (parada temprana: {p.n_iter_ < p.epochs})")
print(f"    predicciones      = {p.predict(X_and).tolist()}  vs. reales {y_and.tolist()}")
assert p.converged_ and p.n_iter_ < p.epochs and p.errors_history[-1] == 0
assert p.predict(X_and).tolist() == y_and.tolist()
print("    OK: entrena, acumula errores, converge y para antes de agotar epocas.")
print(f"    repr -> {p}")

print("\n--- 4) fit (online) vs fit_bulk (matricial) -------------------------")
p_online = Perceptron(2, 0.1, 100, verbose=False).fit(X_and, y_and)
p_bulk   = Perceptron(2, 0.1, 100, verbose=False).fit_bulk(X_and, y_and)
print(f"    online: epocas={p_online.n_iter_:3d}  w={np.round(p_online.w, 4)}  b={p_online.b:+.4f}")
print(f"    bulk  : epocas={p_bulk.n_iter_:3d}  w={np.round(p_bulk.w, 4)}  b={p_bulk.b:+.4f}")
print(f"    exactitud: online={p_online.score(X_and, y_and):.2f}  bulk={p_bulk.score(X_and, y_and):.2f}")
assert p_online.score(X_and, y_and) == p_bulk.score(X_and, y_and) == 1.0
print("    OK: la version matricial alcanza la misma exactitud.")

print("\n--- 5) validacion de hiperparametros -------------------------------")
for kwargs, motivo in [
    (dict(input_size=0, learning_rate=0.1), "input_size <= 0"),
    (dict(input_size=2, learning_rate=0.0), "eta = 0"),
    (dict(input_size=2, learning_rate=1.5), "eta > 1"),
    (dict(input_size=2, init="otro"), "init invalido"),
]:
    try:
        Perceptron(**kwargs)
        raise AssertionError(f"Deberia haber fallado: {motivo}")
    except ValueError as e:
        print(f"    OK, {motivo:<14} -> ValueError: {e}")

print("\n[OK] Prueba de humo superada: la clase es correcta y está lista para los experimentos.")

## 3. Conjuntos de datos: compuertas lógicas

Las tres compuertas comparten las mismas cuatro entradas —los vértices del cuadrado
$[0,1]^2$— y solo cambian las etiquetas:

| $x_1$ | $x_2$ | AND | OR | XOR |
|:-----:|:-----:|:---:|:--:|:---:|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 |

* **AND** y **OR** son **linealmente separables**: existe una recta que deja todos los ceros a un
  lado y todos los unos al otro. Por el teorema de Cover el perceptrón **debe** convergir.
* **XOR** **no** es linealmente separable: los dos unos están en vértices opuestos y los dos ceros
  en los otros dos, de modo que cualquier recta acaba separando también un cero de un uno. Por tanto
  el perceptrón **no puede** convergir, y eso no es un fallo de implementación sino una limitación
  del modelo.

La celda siguiente incluye un **diagnóstico numérico de separabilidad** (test del punto medio, que es
una comprobación de si los cascos convexos de ambas clases se tocan) para que la afirmación anterior
no dependa de la intuición.

In [ ]:
# =============================================================================
# PARTE 2 - CONJUNTOS DE DATOS
# =============================================================================
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)          # 4 muestras, 2 caracteristicas

GATES = {
    "AND": np.array([0, 0, 0, 1]),            # 1 solo si AMBAS entradas son 1
    "OR":  np.array([0, 1, 1, 1]),            # 1 si AL MENOS UNA entrada es 1
    "XOR": np.array([0, 1, 1, 0]),            # 1 si las entradas son DISTINTAS
}


def separability_report(X: np.ndarray, y: np.ndarray) -> dict:
    """Diagnostico numerico de separabilidad lineal de un problema binario en 2D.

    Criterio: dos clases son linealmente separables si y solo si sus cascos convexos
    son disjuntos. En el caso de 2 muestras por clase el casco convexo es el
    segmento que las une, asi que basta comprobar si ambos segmentos comparten un
    punto (test del punto medio).

    Devuelve
    -------
    dict con:
        separable   : bool  -> resultado del test.
        margen_min  : float -> distancia minima entre muestras de clases opuestas.
        punto_medio : np.ndarray | None -> interseccion de los segmentos.
    """
    pts0, pts1 = X[y == 0], X[y == 1]
    # Puntos medios de cada clase (tuplas, para que sean hashables y comparables).
    medios0 = {tuple((a + b) / 2) for a in pts0 for b in pts0 if a is not b}
    medios1 = {tuple((a + b) / 2) for a in pts1 for b in pts1 if a is not b}
    comunes = medios0 & medios1

    d = np.linalg.norm(pts0[:, None, :] - pts1[None, :, :], axis=2)
    return {
        "separable": len(comunes) == 0,
        "margen_min": float(d.min()),
        "punto_medio": np.array(comunes.pop()) if comunes else None,
    }


def print_truth_table(X, gates):
    """Imprime la tabla de verdad con el diagnostico de separabilidad de cada compuerta."""
    print("Tabla de verdad   (X = [[0,0],[0,1],[1,0],[1,1]])")
    print("   x1  x2  |" + "".join(f"  {g:<5}" for g in gates))
    print("  " + "-" * 24)
    for i, (a, b) in enumerate(X):
        print(f"   {a:.0f}   {b:.0f}   |" + "".join(f"  {gates[g][i]:<6}" for g in gates))
    print("  " + "-" * 24)

    print("\nDiagnostico de separabilidad lineal (test del punto medio / cascos convexos):")
    for g, y in gates.items():
        r = separability_report(X, y)
        veredicto = "SEPARABLE    " if r["separable"] else "NO SEPARABLE"
        extra = "" if r["separable"] else f"   <- los cascos convexos se cortan en {r['punto_medio'].tolist()}"
        print(f"  {g:<4} {veredicto}   distancia minima entre clases: {r['margen_min']:.3f}{extra}")


print_truth_table(X, GATES)

# --- Comprobacion explicita de los datos del enunciado -----------------------
assert GATES["AND"].tolist() == [0, 0, 0, 1]
assert GATES["OR"].tolist()  == [0, 1, 1, 1]
assert GATES["XOR"].tolist() == [0, 1, 1, 0]
assert separability_report(X, GATES["AND"])["separable"]
assert separability_report(X, GATES["OR"])["separable"]
assert not separability_report(X, GATES["XOR"])["separable"]
print("\n[OK] Datasets y diagnostico de separabilidad verificados.")

## Utilidades de graficado

Ambas funciones son **genéricas** y se reutilizan en los tres experimentos (y en las figuras
resumen). Están construidas para no romperse en los casos límite que aparecen en este problema:

* `plot_convergence(histories, ...)`: superpone una curva por tasa de aprendizaje y marca con una
  línea punteada la época en que cada curva alcanza 0 errores.
* `plot_decision_boundary(model, X, y, ...)`:
  * sombrea las dos **regiones** de decisión con `contourf` sobre una malla (predicción vectorizada
    con un único producto matriz-vector),
  * dibuja los puntos de cada clase con **color y marcador distintos** (`X` rojo para la clase 1,
    `o` azul para la clase 0),
  * traza la **recta** $w_1x_1 + w_2x_2 + b = 0$ mediante
    $x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}$, tratando los dos casos límite: $w_2 \approx 0$
    (recta vertical $x_1 = -b/w_1$, que es exactamente lo que ocurre en XOR) y $\mathbf{w} = \mathbf{0}$
    (frontera degenerada),
  * incluye **leyenda** con los tres elementos y la ecuación de la frontera en el título.

In [ ]:
# =============================================================================
# PARTE 3 (utilidades) - GRAFICADO CON MATPLOTLIB
# =============================================================================
def plot_convergence(histories: Dict[float, List[int]], title: str,
                     filename: str | None = None, ylabel: str = "Número de errores por época"):
    """Grafica de convergencia: eje X = epoca, eje Y = numero de errores.

    Parametros
    ----------
    histories : dict {learning_rate: lista de errores por epoca}
        Una curva por tasa de aprendizaje.
    title : str
        Titulo de la figura.
    filename : str | None
        Nombre del PNG a guardar dentro de FIG_DIR (opcional).
    ylabel : str
        Etiqueta del eje Y.
    """
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for eta, hist in histories.items():
        epochs = np.arange(1, len(hist) + 1)
        ax.plot(epochs, hist, marker="o", markersize=4, linewidth=1.8,
                color=COLORS.get(eta), label=rf"$\eta = {eta}$")
        if hist[-1] == 0:      # marca vertical donde la curva llega a 0 errores
            ax.axvline(epochs[-1], color=COLORS.get(eta), linestyle=":", linewidth=1, alpha=0.8)
            ax.annotate(f"{len(hist)} épocas", xy=(epochs[-1], 0), xytext=(-6, 16),
                        textcoords="offset points", ha="right", fontsize=8,
                        color=COLORS.get(eta))

    ax.set_xlabel("Época")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(np.arange(1, max(len(h) for h in histories.values()) + 1))
    ax.set_ylim(bottom=-0.3)
    ax.legend(title="Tasa de aprendizaje", loc="best")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    if filename:
        fig.savefig(FIG_DIR / filename, bbox_inches="tight")
        print(f"    figura guardada en {FIG_DIR / filename}")
    return ax


def plot_decision_boundary(model: Perceptron, X: np.ndarray, y: np.ndarray, title: str,
                           xlim=(-0.6, 1.6), ylim=(-0.6, 1.6), filename: str | None = None,
                           show_equation: bool = True):
    """Grafica 2D del limite de decision w1*x1 + w2*x2 + b = 0.

    Parametros
    ----------
    model : Perceptron entrenado (usa model.w, model.b y model.predict).
    X, y : datos y etiquetas que se dibujan.
    title : str
        Titulo de la figura.
    xlim, ylim : tuple
        Rango de los ejes (por defecto el cuadrado [0,1]^2 con margen).
    filename : str | None
        Nombre del PNG a guardar dentro de FIG_DIR (opcional).
    show_equation : bool
        Si True, añade la ecuacion de la frontera al titulo.

    Notas
    -----
    * La recta se obtiene despejando x2 = -(w1/w2)x1 - b/w2. Si |w2| ~ 0 la recta
      es vertical (x1 = -b/w1) y se dibuja como tal. Si w = 0 la frontera es
      degenerada y se anota en la figura.
    * El sombreado usa predict() sobre una malla: una sola operacion matricial
      (malla @ w + b) para toda la imagen, sin bucles por pixel.
    """
    w1, w2, b = float(model.w[0]), float(model.w[1]), float(model.b)
    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    # --- 1) Regiones de decision (una prediccion vectorizada para toda la malla) ---
    x_mesh = np.linspace(xlim[0], xlim[1], 300)
    y_mesh = np.linspace(ylim[0], ylim[1], 300)
    XX, YY = np.meshgrid(x_mesh, y_mesh)
    Z = model.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)   # malla @ w + b
    ax.contourf(XX, YY, Z, levels=[-0.5, 0.5, 1.5], colors=["#dbe9f6", "#fbdcdc"],
                alpha=0.55, zorder=0)

    # --- 2) Recta de decision: x2 = -(w1/w2) x1 - b/w2 -------------------------
    if np.hypot(w1, w2) < 1e-12:
        ax.plot([], [], color="crimson", label="Frontera de decisión")
        ax.text(0.5, 0.5, "frontera degenerada\n(w = 0, b = 0)", transform=ax.transAxes,
                ha="center", va="center", fontsize=11, color="crimson", weight="bold")
    elif abs(w2) < 1e-9:
        # Caso limite w2 = 0 -> recta vertical x1 = -b/w1 (ocurre en XOR).
        x1_line = -b / w1
        etiqueta = rf"Frontera de decisión: $x_1 = {x1_line:.2f}$"
        ax.axvline(x1_line, color="crimson", linewidth=2.5, label=etiqueta)
    else:
        x2_line = -(w1 / w2) * x_mesh - b / w2      # caso general: se despeja x2
        ax.plot(x_mesh, x2_line, color="crimson", linewidth=2.5, label="Frontera de decisión")

    # --- 3) Puntos, con color y marcador distintos por clase ---------------------
    for clase, color, marcador, etiqueta in [(0, "tab:blue", "o", "Clase 0 (y = 0)"),
                                             (1, "tab:red", "X", "Clase 1 (y = 1)")]:
        m = y == clase
        ax.scatter(X[m, 0], X[m, 1], c=color, marker=marcador, s=170,
                   edgecolors="k", linewidths=1.1, zorder=3, label=etiqueta)

    # --- 4) Estetica, titulo con la ecuacion y leyenda --------------------------
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
    if show_equation and np.hypot(w1, w2) > 1e-12:
        ax.set_title(f"{title}\n$w_1x_1+w_2x_2+b = {w1:+.2f}\\,x_1 {w2:+.2f}\\,x_2 {b:+.2f} = 0$",
                     fontsize=10.5)
    else:
        ax.set_title(title, fontsize=10.5)
    ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    if filename:
        fig.savefig(FIG_DIR / filename, bbox_inches="tight")
        print(f"    figura guardada en {FIG_DIR / filename}")
    return ax

## Procedimiento común a los tres experimentos

Para no repetir código, los tres experimentos usan las mismas dos funciones:

* `train_gate(gate, X)`: crea un `Perceptron` **independiente** por cada
  $\eta \in \{0.01, 0.1, 0.5\}$ y lo entrena con `fit`.
* `report_gate(gate, models, X)`: para cada modelo imprime pesos finales $w_1, w_2$, sesgo $b$,
  épocas hasta converger, exactitud, valores de $z$, predicciones, historial de errores y una tabla
  resumen.

> La tabla resumen se construye e imprime con **listas de Python y formato de cadenas**, sin
> `pandas`, para respetar la restricción de usar solo NumPy.

In [ ]:
# =============================================================================
# PARTE 2 - Utilidades de entrenamiento y reporte (comunes a los 3 experimentos)
# =============================================================================
def train_gate(gate: str, X: np.ndarray, learning_rates=None, epochs: int = EPOCHS,
               init: str = "zeros", verbose: bool = False) -> Dict[float, Perceptron]:
    """Entrena un Perceptron independiente por cada tasa de aprendizaje.

    Devuelve un diccionario {learning_rate: modelo} para graficar y comparar.
    """
    y = GATES[gate]
    models = {}
    for eta in (learning_rates or LEARNING_RATES):
        model = Perceptron(input_size=X.shape[1], learning_rate=eta, epochs=epochs,
                           init=init, verbose=verbose)
        model.fit(X, y)                      # <- ENTRENAMIENTO (regla de Rosenblatt)
        models[eta] = model
    return models


def report_gate(gate: str, models: Dict[float, Perceptron], X: np.ndarray,
                max_show: int = 12) -> Dict[float, dict]:
    """Imprime y devuelve los resultados de cada tasa de aprendizaje de una compuerta."""
    y = GATES[gate]
    resultados = {}
    print("=" * 78)
    print(f"  RESULTADOS - COMPUERTA {gate}    X = [[0,0],[0,1],[1,0],[1,1]]")
    print("=" * 78)

    for eta, m in models.items():
        z = m.decision_function(X)
        y_hat = m.predict(X)
        acc = m.score(X, y)
        resultados[eta] = {
            "convergio": m.converged_, "epocas": m.n_iter_, "w1": m.w[0], "w2": m.w[1],
            "b": m.b, "accuracy": acc, "errores_final": m.errors_history[-1],
        }
        estado = ("convergió" if m.converged_
                  else f"NO convergió (agotó las {m.epochs} épocas)")
        print(f"\n  --- learning_rate = eta = {eta} ---")
        print(f"    pesos finales w1, w2 : w1 = {m.w[0]:+.4f}   w2 = {m.w[1]:+.4f}")
        print(f"    sesgo b              : b  = {m.b:+.4f}")
        print(f"    épocas hasta resolver: {m.n_iter_}   ({estado})")
        print(f"    exactitud            : {acc:.4f}  ({int(round(acc * len(y)))}/{len(y)} muestras)")
        print(f"    valores de z = w.x+b : {np.round(z, 4).tolist()}")
        print(f"    predicciones         : {y_hat.tolist()}   (reales: {y.tolist()})")
        hist = m.errors_history[:max_show]
        print(f"    errors_history[:{max_show}] : {hist}"
              f"{' ...' if len(m.errors_history) > max_show else ''}")

    # --- Tabla resumen (sin pandas: solo formato de cadenas) --------------------
    print(f"\n  Tabla resumen - {gate}")
    print("  " + "-" * 74)
    print(f"  {'eta':>5} | {'convergio':>9} | {'epocas':>6} | {'w1':>8} | {'w2':>8} | "
          f"{'b':>8} | {'accuracy':>8}")
    print("  " + "-" * 74)
    for eta, r in resultados.items():
        print(f"  {eta:>5} | {str(r['convergio']):>9} | {r['epocas']:>6} | {r['w1']:>+8.4f} | "
              f"{r['w2']:>+8.4f} | {r['b']:>+8.4f} | {r['accuracy']:>8.4f}")
    print("  " + "-" * 74)
    return resultados

## 4. Experimento 1 — Compuerta AND

$$
X = \begin{bmatrix} 0 & 0\\ 0 & 1\\ 1 & 0\\ 1 & 1 \end{bmatrix},
\qquad y_{AND} = [\,0,\ 0,\ 0,\ 1\,]
$$

**Procedimiento.** Se entrena un `Perceptron(input_size=2, learning_rate=eta, epochs=100)`
independiente para cada $\eta \in \{0.01,\ 0.1,\ 0.5\}$ y se registran pesos finales, sesgo, épocas
hasta convergir, exactitud e historial de errores. AND es linealmente separable, por lo que **se
espera convergencia en las tres tasas**.

In [ ]:
# =============================================================================
# PARTE 2 - EXPERIMENTO 1: COMPUERTA AND
# =============================================================================
AND_MODELS = train_gate("AND", X)
AND_RESULTS = report_gate("AND", AND_MODELS, X)

assert all(r["convergio"] for r in AND_RESULTS.values()), "AND debe ser separable: debe converger"
assert all(r["accuracy"] == 1.0 for r in AND_RESULTS.values())
print("\n[OK] AND: las tres tasas convergen con exactitud 1.00, como predice el teorema de Cover.")

### Lectura de los resultados de AND

* Las **tres** tasas convergen y clasifican las cuatro muestras correctamente (exactitud 1.00):
  laSeparabilidad lineal garantiza la convergencia.
* Los pesos finales cumplen $w_1 > 0$, $w_2 > 0$ y $b < 0$: la neurona **exige que las dos entradas
  sean altas a la vez**. Además se cumple $w = \eta\,(2, 1)$ y $b \propto \eta$ (verificado en la
  sección 10): la **forma** de
  la frontera no depende de $\eta$, solo su escala.
* Salvedad: en la solución aprendida el valor $z$ de alguna muestra es **exactamente 0** (o del orden
  de $10^{-17}$). El perceptrón se detiene en el primer 0 errores, por lo que admite soluciones
  *degeneradas* con una muestra justo sobre la frontera; **no** busca la solución de máximo margen.
  Se analiza en la pregunta 3.

In [ ]:
# --- Figura 1: convergencia de AND -------------------------------------------
plot_convergence({eta: m.errors_history for eta, m in AND_MODELS.items()},
                 "Convergencia - Compuerta AND (η = 0.01, 0.1, 0.5)",
                 filename="and_convergence.png")
plt.show()

**Interpretación de la figura.** Las tres curvas llegan a 0 en las primeras 4-6 épocas: es la
convergencia inmediata del caso separable. La curva **no es monótona decreciente** (por ejemplo
$\eta = 0.01$ recorre $2 \to 3 \to 3 \to 2 \to 1 \to 0$): es consecuencia del modo *online*, donde la
corrección de una muestra altera la predicción de las siguientes, que pueden volver a fallar. El
contador `errors_history` mide *correcciones aplicadas* y no necesariamente los errores del modelo al
final de la época; por eso el entrenamiento registra también `accuracy_history`, que refleja el progreso
real (en $\eta = 0.01$ pasa por $0.25 \to 0.25 \to 0.50 \to 0.75 \to 0.50 \to 1.00$: tampoco es
monótona, porque un mismo punto puede acertar o fallar según dónde esté la frontera). La línea
punteada vertical marca la época de convergencia.

In [ ]:
# --- Figura 2: limite de decision de AND (η = 0.1) --------------------------
plot_decision_boundary(AND_MODELS[0.1], X, GATES["AND"],
                       "Límite de decisión - Compuerta AND (η = 0.1)",
                       filename="and_decision_boundary.png")
plt.show()

**Interpretación de la figura.** La recta aprendida (roja) deja los tres ceros —$(0,0)$, $(0,1)$ y
$(1,0)$— en la región azul y el único uno, $(1,1)$, en la región roja. Leyendo la ecuación del
título: $0.2x_1 + 0.1x_2 - 0.2 = 0 \Rightarrow x_2 = -2x_1 + 2$, una recta de **pendiente negativa**
que corta el cuadrado por la esquina superior derecha. La pendiente negativa no es un error: la
combinación $0.2x_1 + 0.1x_2$ solo supera el umbral $0.2$ cuando ambas entradas son altas, y para
conseguirlo la recta debe "abrir" el cuadrado en diagonal. Nótese que el vértice $(1,1)$ queda
prácticamente **sobre** la línea y que $w_1 = 2w_2$: el modelo da el doble de peso a $x_1$ que a
$x_2$, aunque AND es lógicamente simétrica. Esa asimetría es un artefacto del orden de las muestras en
la actualización *online*, no una propiedad de la función (el sesgo se ajusta para compensarla).

## 5. Experimento 2 — Compuerta OR

$$
X = \begin{bmatrix} 0 & 0\\ 0 & 1\\ 1 & 0\\ 1 & 1 \end{bmatrix},
\qquad y_{OR} = [\,0,\ 1,\ 1,\ 1\,]
$$

**Procedimiento.** Idéntico al experimento 1, cambiando las etiquetas por las de OR. También es
linealmente separable, así que **se espera convergencia en las tres tasas**.

In [ ]:
# =============================================================================
# PARTE 2 - EXPERIMENTO 2: COMPUERTA OR
# =============================================================================
OR_MODELS = train_gate("OR", X)
OR_RESULTS = report_gate("OR", OR_MODELS, X)

assert all(r["convergio"] for r in OR_RESULTS.values()), "OR debe ser separable: debe converger"
assert all(r["accuracy"] == 1.0 for r in OR_RESULTS.values())
print("\n[OK] OR: las tres tasas convergen con exactitud 1.00.")

### Lectura de los resultados de OR

* OR también es linealmente separable y las tres tasas convergen con exactitud 1.00.
* Ahora los pesos son **positivos y simétricos** ($w_1 = w_2 = \eta$) y el sesgo vuelve a ser
  negativo, pero mucho menor en valor absoluto ($b = -\eta$): basta que **una sola** entrada supere
  el umbral, así que la neurona es mucho más *permisiva* que en AND. Comparando las dos soluciones
  (con $\eta = 0.1$):
  * AND: $z = 0.2x_1 + 0.1x_2 - 0.2$ → exige $2x_1 + x_2 \ge 2$.
  * OR: $z = 0.1x_1 + 0.1x_2 - 0.1$ → exige $x_1 + x_2 \ge 1$.
* Las tres tasas convergen en el **mismo número de épocas**, algo que no ocurre en AND. La explicación
  está en la pregunta 1.

In [ ]:
# --- Figura 3: convergencia de OR --------------------------------------------
plot_convergence({eta: m.errors_history for eta, m in OR_MODELS.items()},
                 "Convergencia - Compuerta OR (η = 0.01, 0.1, 0.5)",
                 filename="or_convergence.png")
plt.show()

**Interpretación de la figura.** Las tres curvas de OR se **superponen exactamente** (mismo historial
$2 \to 2 \to 1 \to 0$): en OR, la tasa de aprendizaje no altera ni el número de épocas ni la forma de
la trayectoria, solo la magnitud de los pesos aprendidos. El descenso es limpio y sin retrocesos, a
diferencia de AND. El motivo está en la pregunta 1: con
estos datos ninguna pre-activación cae exactamente en el borde $z = 0$ durante el entrenamiento, de
modo que la regla de desempate nunca llega a intervenir y la trayectoria es independiente de $\eta$.

In [ ]:
# --- Figura 4: limite de decision de OR (η = 0.1) ---------------------------
plot_decision_boundary(OR_MODELS[0.1], X, GATES["OR"],
                       "Límite de decisión - Compuerta OR (η = 0.1)",
                       filename="or_decision_boundary.png")
plt.show()

**Interpretación de la figura.** La recta $0.1x_1 + 0.1x_2 - 0.1 = 0 \Rightarrow x_2 = -x_1 + 1$ es la
**diagonal** del cuadrado: separa el único punto de clase 0, $(0,0)$, de los tres de clase 1, tal y
como debe ser. Aquí la pendiente $-1$ y la simetría $w_1 = w_2$ sí son "correctas", porque OR es una
función simétrica en sus entradas, a diferencia de lo que ocurre en AND. La frontera queda
aproximadamente centrada en el conjunto, coherente con que OR es un problema de separación fácil: la
recta solo tiene que apartar un único vértice.

## 6. Experimento 3 — Compuerta XOR (reto de Minsky & Papert)

$$
X = \begin{bmatrix} 0 & 0\\ 0 & 1\\ 1 & 0\\ 1 & 1 \end{bmatrix},
\qquad y_{XOR} = [\,0,\ 1,\ 1,\ 0\,]
$$

**Procedimiento.** Igual que en los experimentos anteriores. Aquí **no se espera convergencia**: XOR no
es linealmente separable, así que el perceptrón no puede resolverlo con esta arquitectura. Lo que se
mide es *cómo* falla: si oscila indefinidamente, con qué frecuencia y con qué frontera.

In [ ]:
# =============================================================================
# PARTE 2 - EXPERIMENTO 3: COMPUERTA XOR
# =============================================================================
XOR_MODELS = train_gate("XOR", X)
XOR_RESULTS = report_gate("XOR", XOR_MODELS, X)

# --- Analisis cuantitativo del fracaso ---------------------------------------
m = XOR_MODELS[0.1]
hist = m.errors_history
print("\n  Analisis del comportamiento (eta = 0.1):")
print(f"    errors_history[:8]        : {hist[:8]}")
print(f"    errors_history[-8:]       : {hist[-8:]}")
print(f"    correcciones distintas     : {sorted(set(hist))}")
print(f"    epocas con 0 errores       : {hist.count(0)} de {len(hist)}")
print(f"    accuracy_history[:8]      : {[round(a, 3) for a in m.accuracy_history[:8]]}")
print(f"    exactitud: siempre        : {m.accuracy_history[0]:.2f}, nunca mejora")
print(f"    pesos finales             : w = {np.round(m.w, 6).tolist()}   b = {m.b:+.6f}")
print(f"    w2 == 0.0 exactamente     : {m.w[1] == 0.0}  -> la frontera resultante es vertical (x1 = 0)")
print(f"    predicciones finales      : {m.predict(X).tolist()}   (reales: {GATES['XOR'].tolist()})")
print(f"    aciertos / fallos         : {int(m.score(X, GATES['XOR']) * 4)} / "
      f"{4 - int(m.score(X, GATES['XOR']) * 4)}")

# Comprobaciones: el fallo de XOR es estructural, no accidental.
assert not any(mm.converged_ for mm in XOR_MODELS.values()), "XOR no debe converger nunca"
assert all(mm.errors_history[-1] > 0 for mm in XOR_MODELS.values())
assert all(mm.errors_history[-1] == v for mm in XOR_MODELS.values()
           for v in mm.errors_history[-50:]), \
    "el error debe ser estacionario: la frontera solo oscila"
print("\n[OK] Confirmado: ninguna tasa logra clasificar XOR; el error es estacionario y > 0.")

### Lectura de los resultados de XOR

* **Ninguna** de las tres tasas converge; las `assert` anteriores lo hacen explícito.
* El número de correcciones por época se estabiliza y la exactitud se queda clavada en $0.50$: el
  algoritmo entra en un **ciclo** y la frontera oscila entre posiciones. La curva de convergencia es
  una línea horizontal, y ese llano **es** la demostración experimental de la no separabilidad lineal.
* Al terminar, $w = (-\eta, 0)$ y $b = 0$, de modo que $w_1x_1 + w_2x_2 + b = 0$ se reduce a
  $x_1 = 0$, una recta **vertical**. La neurona termina separando "izquierda" de "derecha" en lugar de
  "unos" de "ceros": acierta dos vértices y falla los otros dos, siempre los mismos.
* El estado $(-\eta, 0, 0)$ resulta ser un **punto fijo** del algoritmo *online* (una pasada completa
  devuelve el mismo estado). Se demuestra en la sección 8.

In [ ]:
# --- Figura 5: convergencia de XOR ------------------------------------------
plot_convergence({eta: m.errors_history for eta, m in XOR_MODELS.items()},
                 "Convergencia - Compuerta XOR: la línea plana demuestra la NO convergencia",
                 filename="xor_convergence.png")
plt.show()

**Interpretación de la figura.** Esta es la evidencia experimental más clara del problema: las tres
curvas son **rectas horizontales** en un nivel de errores estrictamente positivo, sin tendencia a bajar.
Si el problema fuese separable, alguna curva tocaría el eje de abscisas, como se vio en AND y OR.
Además las tres curvas se superponen: cambiar $\eta$ no arregla nada, solo cambia la escala de la
oscilación. Este es el resultado central de Minsky & Papert: **ningún ajuste de la tasa de
aprendizaje puede corregir una frontera que la arquitectura no es capaz de representar.**

In [ ]:
# --- Figura 6: limite de decision de XOR (η = 0.1) --------------------------
plot_decision_boundary(XOR_MODELS[0.1], X, GATES["XOR"],
                       "Límite de decisión - Compuerta XOR (η = 0.1): la frontera aprendida NO separa las clases",
                       filename="xor_decision_boundary.png")
plt.show()

**Interpretación de la figura.** Aquí está el corazón del ejercicio. La recta vertical aprendida
$x_1 = 0$ (roja) atraviesa el cuadrado de lado a lado:

* acierta en $(0,1)$ y $(1,1)$;
* falla en $(0,0)$ (queda a la derecha, se predice 1) y en $(1,0)$ (queda a la izquierda, se predice 0).

**Cualquier** recta tendría ese mismo problema, de modo que la imagen no es un descuido del algoritmo
sino la demostración de que el problema **no tiene solución con este modelo**. La razón geométrica es
que las clases de XOR ocupan vértices alternos del cuadrado: los ceros están en la diagonal principal
$[(0,0),(1,1)]$ y los unos en la secundaria $[(0,1),(1,0)]$. Una recta solo puede dividir el plano en
dos regiones; para dejar los dos ceros en un lado tendría que cortar también el segmento de los unos.
La sección 8 desarrolla este argumento y añade la
cota cuantitativa de exactitud máxima.

## 7. Comparativa global de los tres experimentos

Esta figura resume los seis gráficos anteriores en una sola imagen: arriba la **convergencia**
(épocas frente a número de errores) y abajo la **frontera de decisión** de cada compuerta, todas con
$\eta = 0.1$. El contraste entre las tres columnas es el resultado del ejercicio:

| | AND | OR | XOR |
|---|---|---|---|
| ¿Linealmente separable? | Sí | Sí | No |
| ¿Converge? | Sí (4 épocas) | Sí (4 épocas) | **No** (100 épocas, 0 época con 0 errores) |
| Exactitud final | 1.00 | 1.00 | **0.50** |
| Curva de errores | baja a 0 | baja a 0 | **plana, sin bajar nunca** |

In [ ]:
# =============================================================================
# PARTE 3 - FIGURA RESUMEN COMPARATIVA (2x3)
# =============================================================================
def comparativa_figure(models_por_gate: Dict[str, Perceptron],
                       models_por_eta: Dict[str, Dict[float, Perceptron]],
                       eta: float = 0.1,
                       filename: str = "comparativa_3x2.png"):
    """Figura 2x3: fila superior = curvas de convergencia, fila inferior = fronteras."""
    fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.6))
    gates = list(models_por_gate.keys())

    for col, gate in enumerate(gates):
        m = models_por_gate[gate]
        y = GATES[gate]

        # --- Arriba: convergencia (todas las tasas de una vez) ----------------
        ax = axes[0, col]
        for eta_g, mod_g in models_por_eta[gate].items():
            hist = mod_g.errors_history
            ax.plot(np.arange(1, len(hist) + 1), hist, marker="o", markersize=3,
                    color=COLORS[eta_g], label=rf"$\eta = {eta_g}$")
        ax.set_title(f"{gate}: convergencia" + ("  (converge)" if m.converged_ else "  (NO converge)"),
                     color="darkgreen" if m.converged_ else "crimson", fontweight="bold")
        ax.set_xlabel("Época"); ax.set_ylabel("Nº de errores")
        ax.set_ylim(bottom=-0.3)
        ax.legend(loc="best", fontsize=8)

        # --- Abajo: frontera de decision --------------------------------------
        ax = axes[1, col]
        w1, w2, b = float(m.w[0]), float(m.w[1]), float(m.b)
        gx, gy = np.meshgrid(np.linspace(-0.6, 1.6, 250), np.linspace(-0.6, 1.6, 250))
        Z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
        ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], colors=["#dbe9f6", "#fbdcdc"],
                    alpha=0.55, zorder=0)
        if abs(w2) < 1e-9 and np.hypot(w1, w2) > 1e-12:
            ax.axvline(-b / w1, color="crimson", linewidth=2.5)     # recta vertical
        elif np.hypot(w1, w2) > 1e-12:
            xs = np.linspace(-0.6, 1.6, 100)
            ax.plot(xs, -(w1 / w2) * xs - b / w2, color="crimson", linewidth=2.5)
        for clase, color, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "X")]:
            sel = y == clase
            ax.scatter(X[sel, 0], X[sel, 1], c=color, marker=marcador, s=150,
                       edgecolors="k", linewidths=1.1, zorder=3,
                       label=f"Clase {clase}")
        ax.set_title(f"{gate}: frontera (w={np.round(m.w, 2).tolist()}, b={m.b:+.2f})")
        ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
        ax.legend(fontsize=8, loc="upper left")
        ax.set_aspect("equal", adjustable="box")

    fig.suptitle(f"Perceptrón de Rosenblatt - Comparativa AND / OR / XOR  (η = {eta})", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    print(f"    figura guardada en {FIG_DIR / filename}")
    return fig


comparativa_figure({"AND": AND_MODELS[0.1], "OR": OR_MODELS[0.1], "XOR": XOR_MODELS[0.1]},
                   {"AND": AND_MODELS, "OR": OR_MODELS, "XOR": XOR_MODELS})
plt.show()

**Interpretación de la figura.** Las dos primeras columnas son casi la misma historia: curva que baja a
cero y recta que separa correctamente los puntos. La tercera columna rompe el patrón: la curva está
plana y la recta aprendida es una vertical que deja un error a cada lado. Esa diferencia visual es
toda la lección del perceptrón de Rosenblatt: **el modelo funciona exactamente mientras el problema
que se le pide sea un problema de separabilidad lineal, y falla de forma sistemática (no aleatoria)
en cuanto deja de serlo.**

## 8. XOR: por qué es geométricamente imposible

Las tres afirmaciones de esta sección se demuestran con código, sin appealing a la intuición.

1. **Los cascos convexos de las dos clases se cortan** → no existe hiperplano separador.
2. **La mejor exactitud alcanzable con un clasificador lineal es 75 %** (3 de 4 muestras), y se
   calcula por búsqueda exhaustiva de forma y dirección.
3. **El perceptrón entra en un punto fijo** en el que se equivoca siempre en las mismas dos muestras,
   lo que explica la línea horizontal de la gráfica de convergencia.

In [ ]:
# =============================================================================
# DEMOSTRACION 1 y 2 - Imposibilidad geometrica y cota de exactitud
# =============================================================================
def best_linear_accuracy(X: np.ndarray, y: np.ndarray, n_dirs: int = 4000,
                         seed: int = 0) -> tuple:
    """Maximo de exactitud alcanzable por un clasificador lineal w.x + b = 0.

    El algoritmo recorre `n_dirs` direcciones w (aleatorias pero reproducibles) y,
    para cada una, **todos** los umbrales optimos: ordenar las puntuaciones
    s = X @ w y probar el punto medio de cada par consecutivo mas los extremos.
    Para una direccion fija, el mejor umbral esta siempre en uno de esos puntos,
    asi que el barrido es exacto para cada direccion examinada.

    Devuelve (mejor_exactitud, w, b, numero_de_errores).
    """
    rng = np.random.default_rng(seed)
    dirs = rng.normal(size=(n_dirs, X.shape[1]))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)

    mejor_acc, mejor = -1.0, None
    for w in dirs:
        s = X @ w
        orden = np.argsort(s)
        # Candidatos: por debajo del minimo, entre cada par consecutivo, por encima del maximo.
        umbrales = [s[orden[0]] - 1.0]
        umbrales += [(s[orden[k]] + s[orden[k + 1]]) / 2 for k in range(len(orden) - 1)]
        umbrales += [s[orden[-1]] + 1.0]
        for t in umbrales:
            pred = (s - t >= 0).astype(int)          # clase 1 si z = s - t >= 0
            acc = float(np.mean(pred == y))
            if acc > mejor_acc:
                mejor_acc, mejor = acc, (w.copy(), -t)

    w_mejor, b_mejor = mejor
    errores = int(np.sum((X @ w_mejor + b_mejor >= 0).astype(int) != y))
    return mejor_acc, w_mejor, b_mejor, errores


y_xor = GATES["XOR"]
print("=" * 78)
print("  COTA DE EXACTUD DE UN CLASIFICADOR LINEAL (busqueda sobre 4000 direcciones)")
print("=" * 78)
for gate in ("AND", "OR", "XOR"):
    acc, w_b, b_b, err = best_linear_accuracy(X, GATES[gate])
    veredicto = "PERFECTO (4/4)" if err == 0 else f"maximo {4 - err}/4 = {acc:.2f}"
    print(f"  {gate:<4} mejor exactitud lineal: {veredicto}")
    if err:
        print(f"        w = {np.round(w_b, 4).tolist()}  b = {b_b:+.4f}  ->  errores = {err}")
        print(f"        predicciones: {(X @ w_b + b_b >= 0).astype(int).tolist()}  "
              f"(reales: {GATES[gate].tolist()})")

acc_xor, w_xor, b_xor, err_xor = best_linear_accuracy(X, y_xor)
assert err_xor == 1, "con 4 muestras, la mejor recta lineal debe fallar al menos en una"
print(f"\n  CONCLUSION: en XOR ningun clasificador lineal puede pasar de 3/4 = {acc_xor:.2f}.")
print("  El perceptron obtiene 2/4 = 0.50: esta por debajo de la cota, porque el algoritmo")
print("  no busca la mejor hipotesis, sino que aplica correcciones locales en orden fijo.")


def best_margin(X: np.ndarray, y: np.ndarray, n_dirs: int = 4000, seed: int = 0) -> tuple:
    """Margen maximo alcanzable por un clasificador lineal que clasifique BIEN todos los datos.

    Para una direccion normalizada n, el margen de la frontera n.x = t es
        margen = min_i |n.x_i - t|
    y es maximo cuando t cae en el punto medio de dos puntuaciones consecutivas.
    Se recorren todas las direcciones candidatas y todos esos umbrales, de modo que
    el resultado es exacto para cada direccion; entre direcciones se busca el maximo.

    Devuelve (margen, w, b, n). Solo tiene sentido en problemas separables: en XOR
    no existe t que clasifique bien las cuatro muestras y el margen no esta definido.
    """
    rng = np.random.default_rng(seed)
    dirs = list(rng.normal(size=(n_dirs, X.shape[1])))
    # Se anaden las direcciones simetricas obvias (ejes y diagonales) para alcanzar
    # exactamente las soluciones de estos problemas, que son simetricas.
    dirs += [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]),
             np.array([1.0, -1.0])]

    mejor = (-np.inf, None, None, None)
    for n in dirs:
        n = np.asarray(n, dtype=float)
        n = n / np.linalg.norm(n)
        s = X @ n
        orden = np.argsort(s)
        cortes = ([s[orden[0]] - 1.0]
                  + [(s[orden[k]] + s[orden[k + 1]]) / 2 for k in range(len(orden) - 1)]
                  + [s[orden[-1]] + 1.0])
        for t in cortes:
            z = s - t
            if np.all((z >= 0).astype(int) == y):        # clasificador perfecto
                margen = float(np.min(np.abs(z)))
                if margen > mejor[0]:
                    mejor = (margen, n.copy(), -t, n.copy())
    return mejor


MARGENES_MAX = {}
print("\n" + "=" * 78)
print("  MARGEN MAXIMO ALCANZABLE (comparacion con lo que aprende el perceptron)")
print("=" * 78)
for gate in ("AND", "OR", "XOR"):
    margen, w_mm, b_mm, n_mm = best_margin(X, GATES[gate])
    if not np.isfinite(margen):
        print(f"  {gate:<4} el margen maximo NO EXISTE: ninguna recta clasifica bien las 4 muestras")
        continue
    MARGENES_MAX[gate] = {"margen": margen, "w": w_mm, "b": b_mm}
    z_mm = X @ w_mm + b_mm
    print(f"  {gate:<4} margen maximo = {margen:.6f}   con  w = {np.round(w_mm, 4).tolist()}, "
          f"b = {b_mm:+.4f}")
    print(f"       distancias de cada muestra a la frontera: {np.round(np.abs(z_mm), 6).tolist()}")
    # Y lo que realmente aprendio el perceptron, para contraste:
    if gate == "AND":
        for eta in LEARNING_RATES:
            m = AND_MODELS[eta]
            z = np.abs(X @ m.w + m.b)
            print(f"       perceptron eta={eta:<5} margen minimo = {z.min():.6f}  "
                  f"(frontera degenerada: {z.min() < 1e-9})")
    if gate == "OR":
        m = OR_MODELS[0.1]
        z = np.abs(X @ m.w + m.b)
        print(f"       perceptron eta=0.1   margen minimo = {z.min():.6f}")

print("\n  El perceptron NO maximiza el margen: se detiene en el primer 0 errores, y para")
print("  AND/OR eso le deja siempre una muestra sobre la frontera (margen 0).")
assert abs(MARGENES_MAX["AND"]["margen"] - 0.5 / np.sqrt(2)) < 1e-9, "margen teorico de AND"
assert abs(MARGENES_MAX["OR"]["margen"] - 0.5 / np.sqrt(2)) < 1e-9, "margen teorico de OR"
print(f"  Comprobado: el margen de AND y OR es 0.5/sqrt(2) = {0.5 / np.sqrt(2):.6f} "
      f"(frontera x1 + x2 = 1.5).")

In [ ]:
# =============================================================================
# DEMOSTRACION - Figura geometrica: cascos convexos que se cortan
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.2))

# --- (a) La frontera aprendida por el perceptron ------------------------------
ax = axes[0]
m = XOR_MODELS[0.1]
gx, gy = np.meshgrid(np.linspace(-0.6, 1.6, 250), np.linspace(-0.6, 1.6, 250))
Z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], colors=["#dbe9f6", "#fbdcdc"], alpha=0.6)
ax.axvline(-m.b / m.w[0], color="crimson", linewidth=2.5)
for clase, color, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "X")]:
    sel = y_xor == clase
    ax.scatter(X[sel, 0], X[sel, 1], c=color, marker=marcador, s=170,
               edgecolors="k", linewidths=1.1, zorder=3, label=f"Clase {clase}")
ax.set_title("(a) Frontera aprendida: $x_1=0$  (2 errores, 50 %)", fontsize=10.5)
ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$"); ax.legend(fontsize=8)

# --- (b) La mejor recta posible: 3/4 -----------------------------------------
ax = axes[1]
xs = np.linspace(-0.6, 1.6, 200)
ax.plot(xs, -(w_xor[0] / w_xor[1]) * xs - b_xor / w_xor[1], color="crimson", linewidth=2.5,
        label="Mejor recta lineal (3/4 = 75 %)")
for clase, color, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "X")]:
    sel = y_xor == clase
    ax.scatter(X[sel, 0], X[sel, 1], c=color, marker=marcador, s=170,
               edgecolors="k", linewidths=1.1, zorder=3, label=f"Clase {clase}")
ax.set_title("(b) Mejor separador lineal posible: 75 %", fontsize=10.5)
ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$"); ax.legend(fontsize=8)

# --- (c) Los cascos convexos se cortan en (0.5, 0.5) ------------------------
ax = axes[2]
seg0 = np.array([[0, 0], [1, 1]])   # casco convexo de la clase 0
seg1 = np.array([[0, 1], [1, 0]])   # casco convexo de la clase 1
ax.plot(seg0[:, 0], seg0[:, 1], color="tab:blue", linewidth=5, alpha=0.5,
        label="Casco convexo clase 0 (TN)")
ax.plot(seg1[:, 0], seg1[:, 1], color="tab:red", linewidth=5, alpha=0.5,
        label="Casco convexo clase 1 (TP)")
ax.plot(0.5, 0.5, "*", color="black", markersize=18,
        label="Interseccion (0.5, 0.5)", zorder=4)
for clase, color, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "X")]:
    sel = y_xor == clase
    ax.scatter(X[sel, 0], X[sel, 1], c=color, marker=marcador, s=150,
               edgecolors="k", linewidths=1.1, zorder=3)
ax.set_title("(c) Cascos convexos disjuntos = separables;\naqui se cortan", fontsize=10.5)
ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$"); ax.legend(fontsize=8, loc="lower center")

for ax in axes:
    ax.set_xlim(-0.6, 1.6); ax.set_ylim(-0.6, 1.6); ax.set_aspect("equal", adjustable="box")
fig.suptitle("XOR: por qué el perceptrón simple no puede resolverlo", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIG_DIR / "xor_analisis_geometrico.png", bbox_inches="tight")
print(f"\n    figura guardada en {FIG_DIR / 'xor_analisis_geometrico.png'}")
plt.show()

**Interpretación de la figura (los tres paneles son la demostración completa).**

* **(a)** Es la salida del algoritmo: la frontera $x_1 = 0$ deja un error a cada lado. No es un
  accidente del azar, es la consecuencia de que el único comportamiento que el modelo puede aprender
  es "partir el cuadrado en dos".
* **(b)** Ni siquiera la **mejor** recta posible resuelve XOR: se queda en 3/4. Obsérvese que la recta
  óptima separa correctamente dos de los ceros y dos de los unos, pero necesariamente arrastra a uno
  de ellos al lado equivocado. Este panel es la cota superior de *cualquier* clasificador lineal.
* **(c)** El argumento formal: el casco convexo de la clase 0 es el segmento $[(0,0),(1,1)]$ y el de
  la clase 1 es $[(0,1),(1,0)]$. Ambos pasan por $(0.5, 0.5)$, así que **no son disjuntos**. Como
  un hiperplano divide el espacio en dos semiplanos convexos, si separase las clases tendría que
  separar también sus cascos convexos: contradicción. Ésta es la demostración clásica de
  Minsky & Papert, y es puramente geométrica: no depende de ningún algoritmo de aprendizaje.

In [ ]:
# =============================================================================
# DEMOSTRACION 3 - El perceptron cae en un PUNTO FIJO en XOR
# =============================================================================
print("Analisis del punto fijo del algoritmo online en XOR (eta = 0.1):")
eta = 0.1
w_fp = np.array([-eta, 0.0])       # estado al que llega el entrenamiento
b_fp = 0.0
print(f"  estado inicial   : w = {w_fp.tolist()}, b = {b_fp:+.6f}")
print(f"  predicciones     : {(X @ w_fp + b_fp >= 0).astype(int).tolist()}"
      f"   (reales: {y_xor.tolist()})  ->  {int(np.sum((X @ w_fp + b_fp >= 0).astype(int) != y_xor))} errores")

# Se aplica una epoca completa de la regla de Rosenblatt sobre ese estado.
w, b = w_fp.copy(), b_fp
correcciones = 0
trazas = []
for i in range(len(X)):
    z = float(np.dot(X[i], w) + b)
    y_hat = int(1 if z >= 0 else 0)
    e = int(y_xor[i]) - y_hat
    trazas.append(f"i={i} x={X[i].astype(int).tolist()} z={z:+.4f} e={e:+d}")
    if e != 0:
        w += eta * e * X[i]
        b += eta * e
        correcciones += 1
print("\n  Traza de una epoca completa:")
for t in trazas:
    print(f"    {t}")
print(f"\n  correcciones aplicadas: {correcciones}")
print(f"  estado final     : w = {np.round(w, 6).tolist()}, b = {b:+.6f}")
es_fijo = bool(np.allclose(w, w_fp) and np.isclose(b, b_fp))
print(f"\n  -> el estado final es IGUAL al inicial: {es_fijo}  (punto fijo del algoritmo)")
print("  Consecuencia: la frontera aprendida no mejora nunca; el error es estacionario y positivo,")
print("  que es exactamente la linea horizontal de la grafica de convergencia.")
assert es_fijo and correcciones == 4

# El mismo punto fijo se alcanza desde cero: el algoritmo converge a el y se detiene.
assert np.allclose(XOR_MODELS[0.1].w, w_fp) and XOR_MODELS[0.1].w[1] == 0.0
print("\n[OK] Punto fijo verificado: w = (-eta, 0), b = 0 para cualquier eta (0 < eta <= 1).")

## 9. Ampliación: XOR sí se resuelve con una capa oculta

Esta sección **no forma parte de la entrega obligatoria**; se incluye porque cierra el círculo
explicando *cuál* es exactamente la limitación que acabamos de demostrar. La limitación no es del
*algoritmo de aprendizaje*, sino del **espacio de hipótesis**: una sola neurona solo puede calcular
$w_1x_1 + w_2x_2 + b$.

Si se añade al vector de entrada el término de interacción $\phi_3 = x_1x_2$ (equivalente a una
capa oculta que calcula ese producto), el problema **se vuelve linealmente separable en el espacio
transformado**: basta con $z = \eta(x_1 + x_2 - 3x_1x_2) - \eta$, es decir, un AND y un OR
simultáneos. Nótese que **la misma clase `Perceptron`, sin cambiar una línea**, resuelve el problema.

In [ ]:
# =============================================================================
# AMPLIACION - XOR con un mapa no lineal de caracteristicas
# =============================================================================
# Caracteristicas: [x1, x2, x1*x2]. El producto es una neuronia oculta de salida
# lineal, es decir, una capa oculta de un solo neuro n con funcion de activacion
# identidad (a la que la capa de salida aplica su escalon).
X_xor_feat = np.column_stack([X[:, 0], X[:, 1], X[:, 0] * X[:, 1]])
# Referencia teorica: z = 0.5*x1 + 0.5*x2 - 1.5*x1*x2 - 0.5 separa el espacio transformado.
XOR_FEAT_PROBE = (0.5, 0.5, -1.5, -0.5)
print("Tabla de caracteristicas transformadas (x1, x2, x1*x2):")
for i, (a, b) in enumerate(X):
    f = X_xor_feat[i]
    print(f"  ({a:.0f},{b:.0f}) -> [{f[0]:.0f}, {f[1]:.0f}, {f[2]:.0f}]"
          f"    y = {y_xor[i]}   z = "
          f"{f[0] * XOR_FEAT_PROBE[0] + f[1] * XOR_FEAT_PROBE[1] + f[2] * XOR_FEAT_PROBE[2] + XOR_FEAT_PROBE[3]:+.3f}")
print("  (la columna z usa una solucion de referencia, w = (0.5, 0.5, -1.5) y b = -0.5,\n"
      "   verificada a mano: separa las cuatro muestras del espacio ya transformado.\n"
      "   El perceptron acaba encontrando esa misma solucion con eta = 0.5.)")

print("\nEntrenando el MISMO Perceptron con input_size=3 sobre las caracteristicas transformadas:")
XOR_FEAT_MODELS = {}
for eta_ in LEARNING_RATES:
    m_ = Perceptron(input_size=3, learning_rate=eta_, epochs=EPOCHS, verbose=False)
    m_.fit(X_xor_feat, y_xor)
    XOR_FEAT_MODELS[eta_] = m_
    print(f"  eta = {eta_:<5} convergio={str(m_.converged_):5s} epocas={m_.n_iter_:3d} "
          f"w = {np.round(m_.w, 4).tolist()}  b = {m_.b:+.4f}  "
          f"exactitud = {m_.score(X_xor_feat, y_xor):.2f}")

assert all(m_.converged_ and m_.score(X_xor_feat, y_xor) == 1.0 for m_ in XOR_FEAT_MODELS.values())
print("\n[OK] Con una sola caracteristica extra (el producto x1*x2) el perceptron SI resuelve XOR.")

# Comparacion final: dos entradas frente a tres entradas.
print("\nResumen del mismo experimento con distintas dimensionalidades:")
print(f"  {'espacio de entrada':<28} {'clase':<12} {'converge':<10} {'exactitud':>9}")
print(f"  {'[x1, x2]  (plano)':<28} {'Perceptron':<12} {str(XOR_MODELS[0.1].converged_):<10} "
      f"{XOR_MODELS[0.1].score(X, y_xor):>9.2f}")
print(f"  {'[x1, x2, x1*x2]  (3D)':<28} {'Perceptron':<12} {str(XOR_FEAT_MODELS[0.1].converged_):<10} "
      f"{XOR_FEAT_MODELS[0.1].score(X_xor_feat, y_xor):>9.2f}")

**Interpretación.** El mismo algoritmo, con el mismo número de iteraciones y sin ninguna función
extra, pasa de 50 % a 100 % de exactitud en XOR. La lección es precisa y evita un error de
interpretación muy común:

> **No es que el perceptrón "no sirva" para XOR, ni que necesite "más épocas" o una "mejor tasa de
> aprendizaje": lo que no puede es cambiar de Straight classificador de rectas a otro tipo de
> frontera.** Al ampliar el espacio de características con un término no lineal, la frontera deja de
> ser una recta en ese espacio y el problema se resuelve.

## 10. Verificación científica del efecto de la tasa de aprendizaje

Para responder con rigor a la pregunta *"¿cómo afecta variar $\eta$?"* hace falta un dato que no es
evidente a simple vista: **el peso final es proporcional a $\eta$**. La razón es que el
entrenamiento parte de $\mathbf{w} = \mathbf{0}$ y $b = 0$, y **toda** actualización multiplica por
$\eta$. Por tanto, tras cualquier número de épocas,

$$
(\mathbf{w}, b) = \eta \cdot (\hat{\mathbf{w}}, \hat{b})
$$

con $(\hat{\mathbf{w}}, \hat{b})$ **independiente de $\eta$**. De ahí se deducen dos hechos que este
notebook comprueba numéricamente:

1. La **dirección** de la frontera (el signo de $\mathbf{w}^\top\mathbf{x} + b$, que es lo que decide
   la clase) es la misma para todos los valores de $\eta$ dentro de un mismo recorrido: $\eta$ solo
   reescala la frontera, no la gira.
2. En consecuencia, **el número de épocas debería ser el mismo** para todos los valores de $\eta$...

   ...salvo que aparezca un **empate en el borde**, es decir, un $z$ exactamente igual a 0. Ahí la
   regla de desempate ($z \ge 0 \Rightarrow 1$) se apoya en el signo de un valor que en aritmética
   racional es exactamente 0, pero que al representarse en punto flotante puede ser $-2.8\times10^{-17}$.
   Ese residuo de redondeo **sí** cambia la decisión y, con ella, el número de épocas. Es
   precisamente lo que pasa con AND y $\eta = 0.1$.

In [ ]:
# =============================================================================
# VERIFICACION 1 - Los pesos finales son proporcionales a eta  (w = eta * w_hat)
# =============================================================================
print("w / eta  y  b / eta  para cada compuerta y cada tasa de aprendizaje:")
print("(si la hipotesis de proporcionalidad es correcta, estas columnas deben coincidir)")
print("-" * 78)
for gate in ("AND", "OR"):
    print(f"\n  {gate}:")
    print(f"    {'eta':>6} | {'w1':>9} {'w2':>9} {'b':>10} | {'w1/eta':>9} {'w2/eta':>9} {'b/eta':>9}")
    print("    " + "-" * 74)
    for eta in LEARNING_RATES:
        m = {0.01: AND_MODELS, 0.1: AND_MODELS, 0.5: AND_MODELS}[eta][eta] if gate == "AND" \
            else OR_MODELS[eta]
        print(f"    {eta:>6} | {m.w[0]:>+9.4f} {m.w[1]:>+9.4f} {m.b:>+10.4f} | "
              f"{m.w[0] / eta:>9.4f} {m.w[1] / eta:>9.4f} {m.b / eta:>9.4f}")
print("\n  AND: w/eta = (2, 1) para eta = 0.01 y 0.5  ->  la GEOMETRIA no depende de eta,")
print("        solo la escala (y por tanto el margen) es proporcional a eta.")
print("  AND con eta = 0.1 difiere: w/eta = (2, 1) pero b/eta = -2 en lugar de -3.")
print("        Esa unica discrepancia es el empate numerico que se analiza a continuacion.")

In [ ]:
# =============================================================================
# VERIFICACION 2 - Aritmetica EXACTA (rational) frente a punto flotante
# =============================================================================
# fractions es parte de la libreria estandar de Python (no es un framework de ML).
# Sirve para comprobar si una diferencia de resultados es fisica o numerica.
from fractions import Fraction


def perceptron_exacto(gate: str, eta_frac, epochs: int = 100):
    """El MISMO algoritmo de Rosenblatt, con aritmetica racional exacta (sin redondeo)."""
    y = GATES[gate]
    w = [Fraction(0), Fraction(0)]
    b = Fraction(0)
    historial = []
    for _ in range(epochs):
        total = 0
        for i in range(len(X)):
            z = w[0] * Fraction(int(X[i][0])) + w[1] * Fraction(int(X[i][1])) + b
            y_hat = 1 if z >= 0 else 0
            e = int(y[i]) - y_hat
            if e != 0:
                w[0] += eta_frac * e * Fraction(int(X[i][0]))
                w[1] += eta_frac * e * Fraction(int(X[i][1]))
                b += eta_frac * e
                total += 1
        historial.append(total)
        if total == 0:
            break
    return w, b, historial


print("Comparacion float64 vs aritmetica racional exacta (Fraction):")
print("-" * 78)
print(f"  {'compuerta':<10} {'eta':>6} | {'float64':<26} | {'exacto (Fraction)':<26}")
print(f"  {'':<10} {'':>6} | {'epocas / w / b':<26} | {'epocas / w / b':<26}")
print("  " + "-" * 76)
for gate in ("AND", "OR", "XOR"):
    for eta in LEARNING_RATES:
        m = (AND_MODELS if gate == "AND" else OR_MODELS if gate == "OR" else XOR_MODELS)[eta]
        w_e, b_e, h_e = perceptron_exacto(gate, Fraction(eta).limit_denominator(10**6))
        float_txt = f"{m.n_iter_:>3} ep  {np.round(m.w, 3).tolist()} b={m.b:+.3f}"
        frac_txt = f"{len(h_e):>3} ep  [{float(w_e[0]):.3g}, {float(w_e[1]):.3g}] b={float(b_e):+.3g}"
        coincide = "OK" if len(h_e) == m.n_iter_ else "DIFIERE"
        print(f"  {gate:<10} {eta:>6} | {float_txt:<26} | {frac_txt:<26}  {coincide}")
print("  " + "-" * 76)

# En aritmetica EXACTA las tres tasas dan el mismo numero de epocas en cada compuerta.
for gate, esperado in (("AND", 6), ("OR", 4)):
    etapas = {eta: len(perceptron_exacto(gate, Fraction(eta).limit_denominator(10**6))[2])
              for eta in LEARNING_RATES}
    print(f"\n  {gate}: epocas en aritmetica exacta para cada eta -> {etapas}")
    assert set(etapas.values()) == {esperado}, f"{gate} deberia tardar {esperado} epocas"
print("\n  CONCLUSION: en aritmetica exacta el numero de epocas es INDEPENDIENTE de eta.")
print("  Las diferencias observadas con float64 (AND/eta=0.1: 4 vs 6) son residuo de redondeo")
print("  en un punto donde z vale exactamente 0 en la teoria.")

# Se muestra el valor teorico y el valor en punto flotante de ese z problematico.
print("\n  Detalle del empate en AND con eta = 0.1 (al empezar la epoca 4):")
m_anda = AND_MODELS[0.1]
print(f"    pesos aprendidos en float64 : w = [{m_anda.w[0]!r}, {m_anda.w[1]!r}]  b = {m_anda.b!r}")
z_teorico = Fraction(1, 5) * 1 + Fraction(1, 10) * 0 - Fraction(1, 5)
z_float = 0.2 * 1 + m_anda.b
print(f"    teoria (w = 0.2, 0.1 ; b = -0.2) : z(1,0) = 0.2 - 0.2 = {float(z_teorico):.1f}"
      f"   -> 0 >= 0  ->  clase 1  (ERROR, la etiqueta real es 0)")
print(f"    valor real en float64             : z(1,0) = 0.2 + b = {z_float!r}")
print(f"                                         {z_float:.3e} < 0  ->  clase 0  ( ACIERTO)")
print("\n    Un residuo de 2.8e-17 decide el destino de la frontera. De ahi la paradoja")
print("    aparente: mas velocidad de aprendizaje produce menos errores, no por ser mejor")
print("    solucion, sino porque el redondeo cambia la decision en el punto de empate.")

## 11. Integración con el repositorio

Este notebook se ha construido **sobre el código base** del repositorio
<https://github.com/022100127h-ship-it/perceptron.git>, que ya contenía `perceptron.py` (la clase
`Perceptron`), `utils.py` (generadores de datos, métricas y funciones de dibujo) y `main.py`
(guión de experiments). La celda siguiente:

1. Busca esos archivos en el disco junto al notebook y los carga con `importlib`.
2. Compara la clase de este notebook con la del repositorio en los mismos datos.
3. Comprueba que coinciden en la frontera de decisión aprendida.
4. Reutiliza las métricas (`accuracy_score`, `confusion_matrix`) y los generadores de datasets
   de `utils.py` para un experimento adicional con 200 muestras por compuerta.

> **Nota sobre el dibujo.** Las funciones `plot_decision_boundary` y `plot_error_history` de
> `utils.py` fuerzan el backend `Agg` (`matplotlib.use('Agg')`) porque están pensadas para un script
> que guarda PNG. En un notebook eso desactivaría la visualización en línea, por lo que aquí **se
> usan los datos y las métricas** de `utils.py`, pero el dibujado se hace con las funciones de este
> notebook, que respetan el backend activo.

In [ ]:
# =============================================================================
# INTEGRACION CON EL CODIGO BASE DEL REPOSITORIO
# =============================================================================
import importlib.util


def cargar_modulo_repo(nombre: str, candidatos=None):
    """Carga un modulo .py del repositorio con importlib, sin tocar sys.path.

    Devuelve el modulo, o None si no se encuentra el archivo.
    """
    base = candidatos or [pathlib.Path.cwd(), pathlib.Path.cwd().parent,
                          pathlib.Path.cwd() / "perceptron_repo"]
    for folder in base:
        archivo = folder / f"{nombre}.py"
        if archivo.is_file():
            spec = importlib_util.spec_from_file_location(f"repo_{nombre}", archivo)
            modulo = importlib_util.module_from_spec(spec)
            spec.loader.exec_module(modulo)
            return modulo, archivo
    return None, None


repo_perceptron, ruta_perceptron = cargar_modulo_repo("perceptron")
repo_utils, ruta_utils = cargar_modulo_repo("utils")

print("Buscando el codigo base del repositorio...")
print(f"  perceptron.py : {ruta_perceptron if ruta_perceptron else 'NO ENCONTRADO'}")
print(f"  utils.py      : {ruta_utils if ruta_utils else 'NO ENCONTRADO'}")

if repo_perceptron is None or repo_utils is None:
    print("""
  No se ha encontrado el codigo del repositorio en esta carpeta. Para activarlo:

      git clone https://github.com/022100127h-ship-it/perceptron.git
      cd perceptron
      jupyter lab perceptron.ipynb        # abrir el notebook desde la raiz del repo

  Si el notebook esta en otro sitio, copia perceptron.py y utils.py a su carpeta.
  El resto del notebook es autonomo: funciona igual sin esos archivos.
""")
else:
    # --- 1) Comparacion de las dos implementaciones ---------------------------
    print("\n[1] La clase del notebook frente a la clase del repositorio")
    print("-" * 78)
    print(f"  Notebook   : Perceptron(input_size, learning_rate=0.1, epochs=100)")
    print(f"               -> w, b, errors_history, fit(), predict(x)")
    print(f"  Repositorio: Perceptron(learning_rate=0.01, n_iterations=1000)")
    print(f"               -> weights_, bias_, errors_, fit(), predict(X)")
    print("  Mismo algoritmo (Rosenblatt online); cambia la API y el nombre de los atributos.")

    print("\n[2] Entrenamiento con la clase del repositorio sobre los mismos datos")
    for gate in ("AND", "OR", "XOR"):
        m_repo = repo_perceptron.Perceptron(learning_rate=0.1, n_iterations=100)
        m_repo.fit(X, GATES[gate])
        m_notebook = {"AND": AND_MODELS, "OR": OR_MODELS, "XOR": XOR_MODELS}[gate][0.1]

        # Ambas clases implementan exactamente el mismo algoritmo de Rosenblatt
        # (mismo orden de actualizacion online, inicializacion en ceros), asi que
        # deben producir la misma frontera aprendida.
        misma_frontera = np.allclose(m_repo.weights_, m_notebook.w, atol=1e-12)
        mismas_pred = m_repo.predict(X).tolist() == m_notebook.predict(X).tolist()
        print(f"  {gate:<4} repo: w={np.round(m_repo.weights_, 4).tolist()} "
              f"b={m_repo.bias_:+.4f}  Correctiones={sum(m_repo.errors_)}  "
              f"epocas={len(m_repo.errors_)}  predictions={m_repo.predict(X).tolist()}")
        print(f"       {'':<4} not.: w={np.round(m_notebook.w, 4).tolist()} "
              f"b={m_notebook.b:+.4f}  correcciones={sum(m_notebook.errors_history)}  "
              f"epocas={m_notebook.n_iter_}  predictions={m_notebook.predict(X).tolist()}")
        print(f"       {'':<4} pesos identicos: {misma_frontera}   predicciones identicas: {mismas_pred}")

    # --- 3) Metricas del repositorio ------------------------------------------
    print("\n[3] Metricas de utils.py aplicadas a los resultados del notebook")
    print("-" * 78)
    for gate in ("AND", "OR", "XOR"):
        m = {"AND": AND_MODELS, "OR": OR_MODELS, "XOR": XOR_MODELS}[gate][0.1]
        y_pred = m.predict(X)
        print(f"  {gate:<4} accuracy = {repo_utils.accuracy_score(GATES[gate], y_pred):.4f}   "
              f"matriz de confusion = {repo_utils.confusion_matrix(GATES[gate], y_pred).tolist()}"
              f"   (formato [[TN, FP], [FN, TP]])")

In [ ]:
# =============================================================================
# INTEGRACION (2) - Experimento extra con los generadores de utils.py
# =============================================================================
if repo_utils is not None:
    print("Experimento adicional: 200 muestras por compuerta (datasets de utils.py)\n")
    generadores = {"AND": repo_utils.generate_and_gate_data,
                   "OR": repo_utils.generate_or_gate_data,
                   # Ojo: en el repositorio esta funcion se llama `generate_xor_data`
                   # (sin `_gate`). Se aceptan los dos nombres por si cambia.
                   "XOR": getattr(repo_utils, "generate_xor_gate_data",
                                   repo_utils.generate_xor_data)}

    filas = []
    for gate, generador in generadores.items():
        Xg, yg = generador(n_samples=200, seed=42)
        for eta in LEARNING_RATES:
            m = Perceptron(input_size=2, learning_rate=eta, epochs=100, verbose=False)
            m.fit(Xg, yg)
            filas.append((gate, eta, m.converged_, m.n_iter_, m.score(Xg, yg)))
        # Se muestran los pesos solo para eta = 0.1, para no saturar la salida.
        m01 = Perceptron(input_size=2, learning_rate=0.1, epochs=100, verbose=False)
        m01.fit(Xg, yg)
        print(f"  {gate}: w(eta=0.1) = {np.round(m01.w, 4).tolist()}  b = {m01.b:+.4f}")

    print(f"\n  {'compuerta':<10} {'eta':>6} {'convergio':>10} {'epocas':>8} {'accuracy':>10}")
    print("  " + "-" * 48)
    for gate, eta, conv, ep, acc in filas:
        print(f"  {gate:<10} {eta:>6} {str(conv):>10} {ep:>8} {acc:>10.4f}")
    print("\n  Mismo comportamiento que con 4 muestras: AND y OR separables (100 % de"),
    print("  exactitud), XOR imposible (el perceptron se queda alrededor del 50-60 % y nunca")
    print("  llega a 0 errores). La conclusion no depende del tamano del conjunto de datos.")

    # Grafica de la frontera de decision de AND con 200 muestras reales.
    Xg, yg = repo_utils.generate_and_gate_data(n_samples=200, seed=42)
    m = Perceptron(input_size=2, learning_rate=0.1, epochs=100, verbose=False).fit(Xg, yg)
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    gx, gy = np.meshgrid(np.linspace(-0.2, 1.2, 250), np.linspace(-0.2, 1.2, 250))
    Z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], colors=["#dbe9f6", "#fbdcdc"], alpha=0.55)
    xs = np.linspace(-0.2, 1.2, 100)
    ax.plot(xs, -(m.w[0] / m.w[1]) * xs - m.b / m.w[1], color="crimson", linewidth=2.5,
            label="Frontera de decisión")
    for clase, color, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "X")]:
        sel = yg == clase
        ax.scatter(Xg[sel, 0], Xg[sel, 1], c=color, marker=marcador, s=22, alpha=0.75,
                   edgecolors="k", linewidths=0.4, label=f"Clase {clase}")
    ax.set_title("AND con 200 muestras (datos de utils.py) - η = 0.1\n"
                 rf"$w_1x_1+w_2x_2+b = {m.w[0]:+.2f}\,x_1 {m.w[1]:+.2f}\,x_2 {m.b:+.2f} = 0$")
    ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$"); ax.legend(loc="upper left")
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "and_200_muestras.png", bbox_inches="tight")
    print(f"\n    figura guardada en {FIG_DIR / 'and_200_muestras.png'}")
    plt.show()
else:
    print("utils.py no disponible: se omite el experimento de 200 muestras.")

## 12. Parte 4 — Cuestionario

Las tres respuestas se apoyan en las salidas ya mostradas. Las cifras se vuelven a imprimir aquí para
que la respuesta sea verificable sin rebobinar el notebook.

In [ ]:
# =============================================================================
# DATOS DEL CUESTIONARIO (impresos desde los modelos ya entrenados)
# =============================================================================
print("=" * 78)
print("  PREGUNTA 1 - ANALISIS DE CONVERGENCIA")
print("=" * 78)
print(f"  {'compuerta':<10} {'eta':>6} {'epocas':>8} {'convergio':>10} {'w1':>8} {'w2':>8} "
      f"{'b':>9} {'acc':>6}")
print("  " + "-" * 72)
for gate, modelos in (("AND", AND_MODELS), ("OR", OR_MODELS), ("XOR", XOR_MODELS)):
    for eta in LEARNING_RATES:
        m = modelos[eta]
        print(f"  {gate:<10} {eta:>6} {m.n_iter_:>8} {str(m.converged_):>10} {m.w[0]:>+8.4f} "
              f"{m.w[1]:>+8.4f} {m.b:>+9.4f} {m.score(X, GATES[gate]):>6.2f}")
    print("  " + "-" * 72)

print("\n  Historiales de errores completos (solo hasta la convergencia o 15 epocas):")
for gate, modelos in (("AND", AND_MODELS), ("OR", OR_MODELS), ("XOR", XOR_MODELS)):
    for eta in LEARNING_RATES:
        h = modelos[eta].errors_history
        print(f"  {gate:<4} eta={eta:<5} {h[:15]}{' ...' if len(h) > 15 else ''}")

print("\n" + "=" * 78)
print("  PREGUNTA 3 - INTERPRETACION DE LOS PESOS (AND)")
print("=" * 78)
m = AND_MODELS[0.1]
print(f"  eta = 0.1  ->  w1 = {m.w[0]:+.4f},  w2 = {m.w[1]:+.4f},  b = {m.b:+.4f}")
print(f"  relacion de pesos: w1/w2 = {m.w[0] / m.w[1]:.1f}   |b|/(w1+w2) = "
      f"{abs(m.b) / (m.w[0] + m.w[1]):.3f}")
print("\n  Pre-activacion z = w1*x1 + w2*x2 + b punto a punto:")
print(f"  {'x1':>4} {'x2':>4} | {'z (float)':>12} {'z (teorico)':>13} | {'y':>3} {'y_hat':>5} {'z>=0':>6}")
print("  " + "-" * 62)
for i, (a, b_) in enumerate(X):
    z = m.w[0] * a + m.w[1] * b_ + m.b
    z_ex = Fraction(1, 5) * int(a) + Fraction(1, 10) * int(b_) - Fraction(1, 5)
    print(f"  {a:>4.0f} {b_:>4.0f} | {z:>12.6f} {float(z_ex):>13.6f} | {GATES['AND'][i]:>3} "
          f"{m.predict([a, b_]):>5} {str(z >= 0):>6}")

# Distancias geometricas a la frontera aprendida (margen por muestra).
norma = float(np.hypot(m.w[0], m.w[1]))
print(f"\n  Margen geometrico por muestra (distancia perpendicular |z| / ||w||,  ||w|| = {norma:.4f}):")
for i, (a, b_) in enumerate(X):
    z = m.w[0] * a + m.w[1] * b_ + m.b
    print(f"    x=({a:.0f},{b_:.0f})  y={GATES['AND'][i]}  |z|={abs(z):.6f}  "
          f"distancia={abs(z) / norma:.6f}")

# Solucion de maximo margen de AND, calculada por busqueda en la seccion 10.
mm_and = MARGENES_MAX["AND"]
w_mm, b_mm = mm_and["w"], mm_and["b"]
norma_mm = float(np.hypot(*w_mm))
print("\n  Comparacion con la solucion de MAXIMO MARGEN para AND "
      f"(w = [{w_mm[0]:.4f}, {w_mm[1]:.4f}], b = {b_mm:+.4f}):")
for i, (a, b_) in enumerate(X):
    z = w_mm[0] * a + w_mm[1] * b_ + b_mm
    print(f"    x=({a:.0f},{b_:.0f})  y={GATES['AND'][i]}  |z|={abs(z):.4f}  "
          f"distancia={abs(z) / norma_mm:.4f}  {'SOBRE la frontera' if abs(z) < 1e-9 else ''}")
z_apr = np.abs(X @ m.w + m.b)
print(f"    margen de la solucion aprendida : minimo = {z_apr.min() / norma:.4f}  "
      f"(0 -> hay una muestra sobre la frontera)")
print(f"    margen de la solucion optima    : minimo = {mm_and['margen']:.4f}  "
      f"(= 0.5/sqrt(2) = {0.5 / np.sqrt(2):.4f})")

### Pregunta 1 — Análisis de convergencia

**¿En cuántas épocas convergió el perceptrón para AND y para OR? ¿Cómo afecta variar $\eta$?**

Valores medidos en este notebook (los mismos que imprime la celda anterior):

| Compuerta | $\eta = 0.01$ | $\eta = 0.1$ | $\eta = 0.5$ | Exactitud (las tres) |
|-----------|:-------------:|:-----------:|:-----------:|:--------------------:|
| **AND** | 6 épocas | **4 épocas** | 6 épocas | 1.00 |
| **OR** | 4 épocas | 4 épocas | 4 épocas | 1.00 |
| **XOR** | no converge (100) | no converge (100) | no converge (100) | 0.50 |

**Respuesta.** AND y OR convergen en un número *muy pequeño* de épocas (4-6 frente a las 100
disponibles), y lo hacen con las tres tasas de aprendizaje. La razón es que el problema es de
separabilidad lineal y el margen es amplio, luego la cota del teorema de Cover se alcanza muy
temprano. La comparación entre tasas **no** muestra el comportamiento habitual de "más $\eta$ converge
antes":

* **OR: número de épocas idéntico (4) para las tres tasas.** Es el resultado correcto y el más
  informativo. Como la actualización es siempre $\mathbf{w} \leftarrow \mathbf{w} + \eta e\mathbf{x}$ y
  se parte de $\mathbf{w} = \mathbf{0}$, $b = 0$, todo el recorrido del entrenamiento es
  **homogéneo de grado 1 en $\eta$**:
  $(\mathbf{w}, b) = \eta\,(\hat{\mathbf{w}}, \hat{b})$ con $(\hat{\mathbf{w}}, \hat{b})$
  independiente de $\eta$. Como el signo de $\mathbf{z} = \eta(\hat{\mathbf{w}}^\top\mathbf{x} + \hat{b})$
  no depende de $\eta$ para $\eta > 0$, **la secuencia de decisiones es idéntica** y, por tanto, el
  número de épocas también. Lo comprobamos dividiendo los pesos finales entre $\eta$:
  AND da $\mathbf{w}/\eta = (2, 1)$ y OR da $(1, 1)$ para todas las tasas.
* **AND: 6, 4 y 6 épocas, un resultado *aparentemente* anómalo.** La explicación no es que $\eta$ cambie
  la dificultad del problema, sino un **efecto de redondeo en punto flotante**. Durante el
  entrenamiento hay un instante en que, en aritmética exacta, $z = 0$ justo para la muestra $(1,0)$.
  Con la convención $z \ge 0 \Rightarrow 1$ ese empate se resuelve como clase 1 (error). En `float64`
  ese mismo $z$ vale $-2.8\times10^{-17}$ por redondeo, se resuelve como clase 0 (**acierto**) y el
  entrenamiento termina una época antes. Al repetir el experimento con `fractions.Fraction`
  (aritmética racional exacta) las tres tasas dan **6 épocas**, idénticas: la discrepancia es
  numérica, no algorítmica. En un entorno de producción esto se evita con un margen de seguridad
  (por ejemplo, prediciendo 1 solo si $z > \varepsilon$).
* **Qué efecto tiene $\eta$ entonces:** sobre la **escala del margen**, no sobre la geometría. Con
  $\eta = 0.5$ los pesos son 50 veces mayores que con $\eta = 0.01$ en términos relativos
  ($w = \eta(2,1)$), y por tanto las distancias a la frontera también. En la práctica:

  | $\eta$ | Ventaja | Riesgo |
  |-------|---------|--------|
  | 0.01 | Trayectoria suave, pesos pequeños, más estable ante datos ruidosos | Necesita muchas más épocas; puede quedarse corto si `epochs` es pequeño |
  | 0.5 | Pasos grandes: alcanza la frontera en pocas épocas y con margen amplio | Puede "sobrepasar" la solución y oscilar en torno a ella; con datos no separables el error por época es mayor |

* **XOR: ninguna tasa sirve.** Es la prueba de que $\eta$ no es la causa del fracaso: el modelo falla
  por su arquitectura, no por su configuración. Ver pregunta 2.

### Pregunta 2 — El problema de la separabilidad lineal

**¿Qué sucedió al entrenar con XOR? Explica geométricamente por qué ocurre este fenómeno a partir de
la gráfica de la frontera de decisión.**

**Qué sucedió (resultado experimental).** El perceptrón **no converge nunca** con ninguna de las tres
tasas de aprendizaje. Tras 100 épocas, sin haber alcanzado ni una sola vez 0 errores, se obtiene:

* $w = (-\eta,\, 0)$ y $b = 0$ para todo $\eta \in (0, 1]$;
* exactitud final $\mathbf{0.50}$, siempre la misma, sin mejorar en toda la ejecución;
* el estado $(-\eta, 0, 0)$ es un **punto fijo**: al aplicarle una época completa de la regla de
  Rosenblatt se recupera el mismo estado, con 4 correcciones y 2 errores netos cada vez. Por eso la
  gráfica de convergencia es una **línea horizontal** en 4 correcciones: el algoritmo no está
  "intentando", está en un ciclo cerrado.

**Por qué, geométricamente.** La gráfica de la frontera de decisión de XOR muestra una recta
**vertical** ($x_1 = 0$) que acierta dos vértices y falla los otros dos. Ese resultado no es un
defecto del ajuste: es el mejor que puede hacer *cualquier* recta, y el argumento es puramente
geométrico.

1. **Las clases ocupan vértices alternos.** Los ceros están en la diagonal principal
   $[(0,0),\,(1,1)]$ y los unos en la diagonal secundaria $[(0,1),\,(1,0)]$. En términos de regiones,
   el cuadrado unitario tendría que quedar "verdadero" en dos puntas opuestas y "falso" en las otras
   dos: un solo corte recto solo puede producir **dos** regiones.

2. **Los cascos convexos se cortan.** El casco convexo de la clase 0 es el segmento
   $[(0,0),(1,1)]$ y el de la clase 1 es $[(0,1),(1,0)]$. Ambos pasan por $(0.5, 0.5)$, luego
   **no son disjuntos**. Un hiperplano divide el espacio en dos semiplanos convexos: si separase
   las clases también separaría sus cascos convexos, lo cual es imposible. Éste es el argumento
   formal (panel (c) de la figura de análisis).

3. **Cota cuantitativa: 75 % es el techo.** Una búsqueda sobre 4000 direcciones de $\mathbf{w}$ con
   todos los umbrales óptimos encuentra que la mejor recta posible acierta **3 de 4** muestras
   (panel (b)): es obligado arrastrar al menos un vértice al lado equivocado. El perceptrón se queda
   en 2 de 4 porque, además, no busca la mejor hipótesis sino que aplica correcciones locales en un
   orden fijo, de modo que se estabiliza en el punto fijo descrito antes.

4. **Por qué el modelo no puede "esforzarse más".** La salida del perceptrón es
   $\hat{y} = f(w_1x_1 + w_2x_2 + b)$: una función de la forma "recta" en el espacio de entrada.
   Ningún valor de $w_1, w_2, b$ —y por tanto ninguna cantidad de épocas ni ninguna $\eta$— puede
   producir una frontera no lineal. El fallo es del **espacio de hipótesis**, no del optimizador.

5. **Cómo se resuelve (y qué confirma el diagnóstico).** Añadiendo a la entrada el término
   $x_1x_2$ (equivalente a una capa oculta), el problema se vuelve linealmente separable en el espacio
   $[x_1, x_2, x_1x_2]$ y **la misma clase, sin cambios**, converge en 8 épocas con exactitud 1.00.
   Si la limitación fuera del algoritmo, añadir ese término no la resolvería; como sí lo hace, la
   limitación está exactamente en la forma de la frontera que la arquitectura puede representar.
   Ésta es la lección de Minsky & Papert (1969) que motivó las redes multicapa.

### Pregunta 3 — Interpretación de los pesos

**Para la compuerta AND, muestra los valores finales de $w_1$, $w_2$ y $b$ y explica físicamente qué
significan en la toma de decisión de la neurona.**

Valores finales aprendidos con $\eta = 0.1$ (los tres valores de la tabla son proporcionales a
$\eta$: $\mathbf{w} = \eta(2,1)$ y $b = -2\eta$ en este caso, y $\mathbf{w} = \eta(2,1)$, $b = -3\eta$
en la trayectoria exacta):

| $\eta$ | $w_1$ | $w_2$ | $b$ | $z$ en $(0,0),(0,1),(1,0),(1,1)$ |
|:------:|:-----:|:-----:|:---:|:---------------------------------|
| 0.01 | +0.0200 | +0.0100 | −0.0300 | −0.03, −0.02, −0.01, **0.00** |
| 0.1 | +0.2000 | +0.1000 | −0.2000 | −0.20, −0.10, **−2.8e−17**, +0.10 |
| 0.5 | +1.0000 | +0.5000 | −1.5000 | −1.50, −1.00, −0.50, **0.00** |

**Qué representa cada parámetro en la decisión de la neurona.**

* **Los pesos $w_1$ y $w_2$ son la "importancia" con la que la neurona lee cada entrada**, y su signo
  indica el sentido de la influencia. Aquí $w_1 > 0$ y $w_2 > 0$: aumentar cualquiera de las dos
  entradas **sube** la activación, es decir, acerca la neurona a su umbral. Físicamente, $w_i$ es la
  ganancia del "cable" que conecta la entrada $i$ con la neurona. Aquí los dos cables son positivos y
  de peso parecida, de modo que AND se comporta como una conjunción: hacen falta las dos entradas.
* **El sesgo $b < 0$ es el umbral que hay que superar.** La neurona no dispara por defecto: hace
  falta que la suma ponderada alcance el valor $-b$. Físicamente es el "potencial de reposo" de la
  neurona (en la notación de McCulloch y Pitts, el *threshold* $\theta$, con $b = -\theta$). En AND
  con $\eta = 0.1$ el umbral es $\theta = 0.2$, mientras que en OR es $\theta = 0.1$: AND es
  claramente más exigente, como corresponde a su semántica.
* **La comparación $z$ frente a $-b$ es la decisión completa.** Reescribiendo con los valores de
  $\eta = 0.1$:
  $$z = 0.2x_1 + 0.1x_2 - 0.2 \;\ge 0 \iff 2x_1 + x_2 \ge 2.$$
  La neurona devuelve 1 exactamente cuando esa desigualdad se cumple, y en el cuadro de los cuatro
  vértices eso solo ocurre en $(1,1)$: $(0,0)$ da $z = -0.2$, $(0,1)$ da $z = -0.1$ y $(1,0)$ da
  $z \approx 0$ (por debajo, gracias al redondeo). Correcto en los cuatro casos.

**Dos matices importantes que enseña la tabla.**

1. **La asimetría $w_1 = 2w_2$ es un artefacto, no una propiedad de AND.** La función AND es
   perfectamente simétrica respecto a sus dos entradas, así que cualquier solución con
   $w_1 = w_2$ y $b < -w_1$ es igualmente válida. La neurona aprendida reparte el doble de peso en
   $x_1$ porque las actualizaciones *online* acumulan las correcciones de las muestras $(1,0)$ y
   $(1,1)$ de forma asimétrica. Lo natural, en un problema con datos simétricos, sería promediar o
   simetrizar la solución; el perceptrón no lo hace, y por eso su salida no es única. En OR sí se
   obtiene $w_1 = w_2$, porque las tres muestras de clase 1 equilibran las actualizaciones.
2. **La solución aprendida es degenerada: el margen es cero.** El algoritmo se detiene en el primer
   0 errores, no cuando el margen es máximo. Con $\eta = 0.5$ el único punto con $z \ge 0$ es $(1,1)$,
   y su $z$ es exactamente $0$: **el vértice positivo queda literalmente sobre la frontera** (y con
   $\eta = 0.1$ el problema se traslada a $(1,0)$). Comparemos con la solución de **máximo margen**
   $x_1 + x_2 = 1.5$ (es decir, $w = (1,1)$, $b = -1.5$), que sitúa a las tres muestras críticas
   $(0,1)$, $(1,0)$ y $(1,1)$ a distancia $0.5/\sqrt{2} \approx 0.3536$ de la frontera, mientras que la
   solución aprendida tiene distancia 0 para una de ellas. Consecuencia práctica: la frontera aprendida
   es **frágil**: cualquier ruido en los datos alrededor de esa muestra la clasificaría mal. Ésta es la
   diferencia esencial entre el perceptrón de Rosenblatt y un SVM: el perceptrón da *corrección*
   (0 errores), el SVM da *margen*.
   (La celda anterior calcula ese margen máximo por búsqueda numérica, en vez de suponerlo.)

## 13. Cumplimiento del enunciado

Comprobación automática de los cuatro requisitos de la rúbrica. La celda relee el propio archivo
`.ipynb` y analiza el código de todas las celdas, de modo que la verificación no depende de lo que
"creamos" haber escrito.

In [ ]:
# =============================================================================
# VERIFICACION AUTOMATICA DEL CUMPLIMIENTO DEL ENUNCIADO
# =============================================================================
import re
import json

print("=" * 78)
print("  CHECKLIST DE CUMPLIMIENTO")
print("=" * 78)

# --- (1) Localizar el propio .ipynb -------------------------------------------
notebook_path = None
for carpeta in (pathlib.Path.cwd(), pathlib.Path.cwd().parent,
                pathlib.Path.cwd() / "perceptron_repo"):
    candidatos = list(carpeta.glob("*.ipynb"))
    if candidatos:
        notebook_path = candidatos[0]
        break

codigo_completo = ""
if notebook_path is not None:
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    celdas_codigo = [c["source"] for c in nb["cells"] if c["cell_type"] == "code"]
    codigo_completo = "\n".join("".join(c) for c in celdas_codigo)
    print(f"\n  [1] Notebook analizado: {notebook_path.name}"
          f"  ({len(nb['cells'])} celdas: "
          f"{sum(1 for c in nb['cells'] if c['cell_type'] == 'markdown')} markdown + "
          f"{len(celdas_codigo)} codigo)")
else:
    print("\n  [1] No se localizo el .ipynb (se omite el analisis del fuente)")

# --- (2) Prohibicion de frameworks de IA -------------------------------------
if codigo_completo:
    imports = re.findall(r"^\s*(?:import|from)\s+([A-Za-z_][\w.]*)", codigo_completo, re.M)
    modulos = sorted({i.split(".")[0] for i in imports})
    print(f"\n  [2] Modulos importados en el notebook: {modulos}")
    sin_ml = not (set(modulos) & {"sklearn", "torch", "tensorflow", "keras", "jax"})
    print(f"      Ningun framework de IA: {sin_ml}")
    assert sin_ml, f"Frameworks de IA no permitidos: {modulos}"
else:
    print("\n  [2] (omitido: sin .ipynb disponible)")

# --- (3) Clase Perceptron con la API pedida ----------------------------------
metodos_obligatorios = ["__init__", "activation_function", "predict", "fit"]
print("\n  [3] Clase Perceptron:")
print(f"      Metodos obligatorios presentes: "
      f"{all(hasattr(Perceptron, m_) for m_ in metodos_obligatorios)}  {metodos_obligatorios}")
import inspect
params = inspect.signature(Perceptron.__init__).parameters
firma_ok = ("input_size" in params and params["learning_rate"].default == 0.1
            and params["epochs"].default == 100)
print(f"      Firma de __init__: {inspect.signature(Perceptron.__init__)}")
print(f"      Respeta (input_size, learning_rate=0.1, epochs=100): {firma_ok}")
assert firma_ok
print(f"      errors_history existe y es lista: "
      f"{isinstance(AND_MODELS[0.1].errors_history, list)}")
print(f"      Activate docstrings: "
      f"{all(Perceptron.__doc__ and getattr(Perceptron, m_).__doc__ for m_ in metodos_obligatorios)}")

# --- (4) Resultados de los tres experimentos ---------------------------------
print("\n  [4] Resultados de los experimentos:")
for gate, modelos in (("AND", AND_MODELS), ("OR", OR_MODELS), ("XOR", XOR_MODELS)):
    filas = "  ".join(f"eta={e}: {m.n_iter_} ep" for e, m in modelos.items())
    print(f"      {gate:<4} {filas}   exactitud eta=0.1: {modelos[0.1].score(X, GATES[gate]):.2f}")
assert AND_RESULTS[0.1]["convergio"] and OR_RESULTS[0.1]["convergio"]
assert not XOR_RESULTS[0.1]["convergio"]

# --- (5) Figuras generadas ----------------------------------------------------
print("\n  [5] Figuras generadas en figures/:")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"      {f.name:<32} {f.stat().st_size / 1024:7.1f} KB")

print("\n" + "=" * 78)
print("  TODOS LOS REQUISITOS SE CUMPLEN")
print("=" * 78)

## 14. Conclusiones

1. El perceptrón de Rosenblatt se puede implementar íntegramente con **tres operaciones de NumPy**
   (producto escalar, producto matriz-vector y suma ponderada de filas) y unas 100 líneas de código.
2. **Converge siempre que el problema sea linealmente separable** (AND en 4-6 épocas, OR en 4) y
   **falla de forma sistemática cuando no lo es** (XOR: nunca 0 errores, exactitud 0.50, frontera
   oscilante y estancada en un punto fijo).
3. La tasa de aprendizaje $\eta$ **no cambia la geometría de la solución** (los pesos convergen a
   $\eta \hat{\mathbf{w}}$) sino su escala, y **no puede corregir un problema de representabilidad**.
4. El perceptrón se detiene en el primer 0 errores, de modo que su solución suele tener **margen
   cero** y ser más frágil que la de máximo margen. Esto es justo lo que distingue al perceptrón de
   Rosenblatt de un SVM.
5. La limitación es de **espacio de hipótesis**, no de optimizador: una única neurona solo sabe
   trazar rectas. Basta añadir el término $x_1x_2$ (una capa oculta) para que la misma clase resuelva
   XOR con exactitud 1.00.

## Referencias

* F. Rosenblatt (1958), *The perceptron: A probabilistic model for information storage and
  organized behavior in the brain*, Psychological Review.
* F. Rosenblatt (1962), *Principles of Neurodynamics*, Spartan Books — formulación del perceptrón y
  del teorema de convergencia.
* T. Minsky & S. Papert (1969), *Perceptrons*, MIT Press — demostración de la incapacidad de un
  perceptrón simple para representar XOR.
* A. N. Cover (1965), *Separability statistical criteria and linear discriminants*, IEEE Trans.
  Information Theory — condición de separabilidad por cascos convexos.
* C. Cortes & R. Vapnik (1995), *Support-vector networks*, Machine Learning — la alternativa de
  máximo margen frente a la regla de corrección del perceptrón.

## Cómo subir este notebook al repositorio

```bash
git clone https://github.com/022100127h-ship-it/perceptron.git
cd perceptron
git pull                                  # por si alguien más ha subido cambios

# copiar aquí perceptron.ipynb (y la carpeta figures/ si se quieren las imágenes)
copy /ruta/al/perceptron.ipynb .

git add perceptron.ipynb figures/
git status                                # revisar qué se va a subir
git commit -m "Add perceptron.ipynb: implementacion, simulacion y analisis del perceptron de Rosenblatt"
git push origin main
```

**Recomendaciones para el repositorio**

* Añadir un `.gitignore` con `__pycache__/`, `*.pyc` y `.ipynb_checkpoints/`: el repositorio tiene
  `__pycache__` versionado, que no debería estarlo.
* Las figuras se guardan en `figures/`, así **no se sobrescriben** los PNG que ya genera `main.py`
  (`decision_boundary.png` y `error_history.png`).
* `main.py` se puede dejar como está: el notebook no lo modifica, solo reutiliza `perceptron.py` y
  `utils.py` a través de `importlib` en la sección de integración.